# Tsunami Quantitative Analysis


## 0. Runtime configuration


In [1]:
from pathlib import Path
import itertools
import json
import math
import platform
import re
import warnings
import zlib
from statistics import NormalDist

import numpy as np
import scipy
import inspect
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
from patsy import build_design_matrices
from IPython.display import display


# Set this to the directory containing the distributed input CSV files.
DATA_DIR = Path('./input')

REQUIRED_CSV_FILES = (
    'city_race_aggregated.csv', 'precision.csv',
    'questionnaire_baseline.csv', 'questionnaire_minimap.csv',
    'questionnaire_tsunami.csv', 'questionnaire_cybersickness.csv',
    'questionnaire_raw-tlx.csv', 'questionnaire_sbsod.csv',
    'questionnaire_demography.csv', 'questionnaire_preferences.csv',
    'city_race_precision_200_to_500m.csv', 'city_race_orientation_events.csv',
    'city_race_checkpoint_approach_episodes.csv',
)

missing_inputs = [name for name in REQUIRED_CSV_FILES if not (DATA_DIR / name).exists()]
if missing_inputs:
    raise FileNotFoundError(f'Missing required input CSV files: {missing_inputs}')

LIBRARY_VERSIONS = {
    'python': platform.python_version(),
    'numpy': np.__version__, 'pandas': pd.__version__,
    'scipy': scipy.__version__ if 'scipy' in globals() else stats.__version__,
    'matplotlib': plt.matplotlib.__version__,
    'seaborn': sns.__version__, 'statsmodels': sm.__version__,
}
display(pd.Series(LIBRARY_VERSIONS, name='Version').to_frame())

LMM_FIT_LOG = []
# Always recover the actual statsmodels formula constructor. This is safe even
# when this setup cell is rerun after an older wrapper was already installed.
_MIXEDLM_FROM_FORMULA = MixedLM.from_formula
ANALYSIS_CONTEXT = {'Hypothesis': '', 'Analysis': ''}

def _infer_lmm_dataset(data):
    columns = set(getattr(data, 'columns', []))
    if {'InitialTargetDistance2D', 'FirstLandingError2D'} <= columns:
        return 'city_race_checkpoint_approach_episodes_analysis'
    if {'DistanceBandBalanced', 'LandingError2D'} <= columns:
        return 'city_race_precision_analysis'
    if {'Distance', 'AimingError'} <= columns:
        return 'precision_analysis'
    if {'PathID', 'NetPathTime'} <= columns:
        return 'city_analysis'
    if {'TargetID', 'BannerDelayTime'} <= columns:
        return 'city_race_orientation_events_analysis'
    return 'derived_analysis_dataset'

class _TrackedMixedLM:
    def __init__(self, model, metadata):
        self._model = model
        self._metadata = metadata

    def __getattr__(self, name):
        return getattr(self._model, name)

    def fit(self, *args, **kwargs):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            fitted = self._model.fit(*args, **kwargs)
        warning_text = ' | '.join(
            f'{item.category.__name__}: {item.message}' for item in caught
        )
        for item in caught:
            if 'boundary of the parameter space' not in str(item.message).lower():
                warnings.warn(str(item.message), item.category, stacklevel=2)
        random_variance = np.nan
        if getattr(fitted, 'cov_re', None) is not None and fitted.cov_re.size:
            random_variance = float(fitted.cov_re.iloc[0, 0])
        LMM_FIT_LOG.append({
            **self._metadata,
            'Converged': bool(getattr(fitted, 'converged', False)),
            'RandomInterceptVariance': random_variance,
            'ResidualVariance': float(getattr(fitted, 'scale', np.nan)),
            'ICC': (random_variance / (random_variance + float(getattr(fitted, 'scale', np.nan)))
                    if np.isfinite(random_variance) and float(getattr(fitted, 'scale', np.nan)) > 0 else np.nan),
            'Warnings': warning_text,
        })
        return fitted

def tracked_mixedlm(formula, *args, **kwargs):
    caller = inspect.currentframe().f_back
    local_values = caller.f_locals
    data = args[0] if args else kwargs.get('data')
    outcome = str(local_values.get('outcome', formula.split('~', 1)[0].strip()))
    transformation = local_values.get('transformation')
    if transformation is None:
        model_outcome = formula.split('~', 1)[0].strip()
        if model_outcome.startswith('Log1p'):
            transformation = 'log1p'
        elif model_outcome.startswith('Log'):
            transformation = 'log'
        elif _infer_lmm_dataset(data) == 'city_race_orientation_events_analysis':
            # These model functions explicitly construct ModelOutcome via np.log1p.
            transformation = 'log1p'
        else:
            transformation = 'identity'
    analysis = ANALYSIS_CONTEXT.get('Analysis', '')
    outcome_role = str(local_values.get('role', ''))
    if not outcome_role:
        outcome_role = 'supportive' if 'support' in analysis.lower() else 'primary'
    has_method_effect = re.search(r'C\(Method(?:,|\))', formula) is not None
    purpose = (
        'interaction' if '*' in formula else
        'reduced' if not has_method_effect else
        'additive'
    )
    groups = kwargs.get('groups')
    metadata = {
        'Analysis': analysis,
        'Hypothesis': ANALYSIS_CONTEXT.get('Hypothesis', ''),
        'Outcome': outcome,
        'OutcomeRole': outcome_role,
        'Transform': transformation,
        'Dataset': _infer_lmm_dataset(data),
        'ModelPurpose': purpose,
        'Formula': formula,
        'RandomEffects': kwargs.get('re_formula', 'random intercept'),
        'NParticipants': int(pd.Series(groups).nunique()) if groups is not None else np.nan,
        'NObservations': int(len(data)) if data is not None else np.nan,
    }
    return _TrackedMixedLM(
        _MIXEDLM_FROM_FORMULA(formula, *args, **kwargs), metadata
    )
smf.mixedlm = tracked_mixedlm

EXPECTED_CURRENT_SAMPLE_SIZE = None  # Observed N is inferred dynamically from the files.
PLANNED_FINAL_SAMPLE_SIZE = 42
RANDOM_SEED = 20260730
ALPHA = 0.05
CONFIDENCE_LEVEL = 0.95
MULTIPLE_COMPARISON_METHOD = 'holm'
EFFECT_BOOTSTRAP_RESAMPLES = 5_000

# Optional smallest effects of practical interest (SESOI), expressed in the
# original outcome units. Leave as None until a defensible threshold is set.
PRACTICAL_THRESHOLDS = {
    'NetPathTime': None, 'TotalPathTime': None, 'BannerDelayTime': None,
    'HeadRotation3D': None, 'SpatialAwarenessComposite': None,
    'environmental_continuity': None, 'CSQTotalChange': None,
    'RAWTLX': None, 'UsabilityComposite': None,
    'DistanceAwarenessComposite': None,
}

METHODS = ('baseline', 'map', 'tsunami')
PRECISION_TARGET_X = {50: 81.7, 100: 31.7, 200: -68.3}
PRECISION_DISTANCE_LABELS = {50: 'Short (50 m)', 100: 'Medium (100 m)', 200: 'Long (200 m)'}
METHOD_LABELS = {'baseline': 'Baseline', 'map': 'Minimap', 'tsunami': 'Tsunami'}
METHOD_COLORS = {'baseline': '#7f7f7f', 'map': '#dd8452', 'tsunami': '#4c72b0'}
ROUTES = ('A', 'B', 'C')
ROUTE_LENGTHS_M = {'A': 2589.5290707671866, 'B': 2411.0317353962837, 'C': 2580.0081858076164}

INPUT_FILES = {
    'city': 'city_race_aggregated.csv', 'precision': 'precision.csv',
    'baseline': 'questionnaire_baseline.csv', 'map': 'questionnaire_minimap.csv',
    'tsunami': 'questionnaire_tsunami.csv', 'csq': 'questionnaire_cybersickness.csv',
    'tlx': 'questionnaire_raw-tlx.csv', 'sbsod': 'questionnaire_sbsod.csv',
    'demography': 'questionnaire_demography.csv', 'preferences': 'questionnaire_preferences.csv',
    'city_precision': 'city_race_precision_200_to_500m.csv',
    'orientation_events': 'city_race_orientation_events.csv',
    'checkpoint_episodes': 'city_race_checkpoint_approach_episodes.csv',
}

METHOD_SEQUENCES = (
    ('baseline', 'map', 'tsunami'), ('map', 'tsunami', 'baseline'),
    ('tsunami', 'baseline', 'map'), ('baseline', 'tsunami', 'map'),
    ('map', 'baseline', 'tsunami'), ('tsunami', 'map', 'baseline'),
)

sns.set_theme(style='whitegrid', context='notebook')


             Version
python       3.12.13
numpy          2.5.1
pandas         3.0.5
scipy         1.18.0
matplotlib    3.11.1
seaborn       0.13.2
statsmodels   0.14.6


## 1. Analysis overview

The analysis is organized by the three research questions. Every result is labelled **hypothesis-directed**, **supplementary**, **exploratory**, or **descriptive**.

- Primary trial- and route-level models retain the datasets, transformations, fixed effects, random effects, reference levels, and planned contrasts of the working notebook.
- All simple participant-level comparisons use the Wilcoxon signed-rank test with `zero_method="wilcox"`, `correction=False`, and `method="auto"`.
- One-sided alternatives are used only for hypotheses with an a-priori direction. The reported difference is always the first named condition minus the second.
- Raw and Holm-adjusted p-values, matched-pairs rank-biserial correlations, raw-unit differences, and confidence intervals support statistical and practical interpretation.
- Paired t-tests appear only in the final sensitivity analysis and never select the primary test.

### Missing City Race observations

**Mixed-effects analyses use all available observations, whereas participant-level analyses requiring aggregation across City Race routes use only participant × method observations with complete route coverage.** Complete route coverage is evaluated separately for each outcome and requires at least one valid, non-missing observation on Routes A, B, and C. An incomplete participant × method summary is omitted only from paired contrasts that require that method; the participant remains eligible for other complete contrasts and for all unrelated study outcomes. Route-specific paired analyses independently retain every participant who has both values required by that route-level contrast. No missing route is imputed.


### Analysis scope and classification

This notebook contains the quantitative analyses of the user study. The qualitative interview analysis was conducted separately and is documented in a separate supplementary package.

The study was not preregistered. The research questions, hypotheses, expected directions, and primary outcomes were formulated before inferential analysis. We therefore describe these analyses as **hypothesis-directed** and distinguish them from supplementary and exploratory analyses. This terminology does not imply preregistration.

Participant-level inference used Wilcoxon signed-rank tests. Rank-biserial correlations and Hodges--Lehmann shifts are reported as corresponding rank-based summaries. Arithmetic means, paired mean differences, their confidence intervals, and Cohen's $d_z$ are descriptive original-scale summaries. Paired $t$-tests are reported only as sensitivity analyses.


## Analysis map

This table links each research question and hypothesis to its notebook section, source dataset, classification, and main figure where applicable.


In [2]:
ANALYSIS_MAP_ROWS = [
    ('RQ1', 'H1.1', 'NetPathTime', 'hypothesis-directed',
     'H1.1 Navigation Task efficiency', 'main_rq1_city_race_times.svg',
     'city_race_aggregated.csv'),
    ('RQ1', 'H1.1', 'TotalPathTime', 'supplementary',
     'Supplementary TotalPathTime', 'main_rq1_city_race_times.svg',
     'city_race_aggregated.csv'),
    ('RQ1', '', 'Precision Task outcomes', 'supplementary',
     'Precision Task', 'main_rq1_precision_200m_landings.svg', 'precision.csv'),
    ('RQ1', '', 'Navigation Task checkpoint precision', 'exploratory',
     'Navigation Task checkpoint precision', '',
     'city_race_precision_200_to_500m.csv'),
    ('RQ2', 'H2.1', 'Behavioral re-engagement', 'hypothesis-directed',
     'H2.1 behavioral re-engagement', 'main_rq2_reorientation_awareness.svg',
     'city_race_orientation_events.csv'),
    ('RQ2', 'H2.2', 'Perceived spatial awareness', 'hypothesis-directed',
     'H2.2 perceived spatial awareness', 'main_rq2_reorientation_awareness.svg',
     'questionnaire_{method}.csv'),
    ('RQ2', '', 'Environmental continuity', 'exploratory',
     'Environmental continuity', 'main_rq2_reorientation_awareness.svg',
     'questionnaire_{method}.csv'),
    ('RQ3', 'H3.1', 'CSQTotalChange', 'hypothesis-directed',
     'H3.1 cybersickness', '', 'questionnaire_cybersickness.csv'),
    ('RQ3', 'H3.2', 'RAWTLX', 'hypothesis-directed',
     'H3.2 workload', '', 'questionnaire_raw-tlx.csv'),
    ('RQ3', '', 'Usability and distance awareness', 'exploratory',
     'Subjective usability and distance awareness', '',
     'questionnaire_{method}.csv'),
    ('RQ3', '', 'Rankings', 'exploratory', 'Rankings', '',
     'questionnaire_preferences.csv'),
]
analysis_map = pd.DataFrame(ANALYSIS_MAP_ROWS, columns=[
    'ResearchQuestion', 'Hypothesis', 'Outcome', 'Classification',
    'NotebookSection', 'Figure', 'SourceCSV',
])
display(analysis_map)


   ResearchQuestion  ...                            SourceCSV
0               RQ1  ...             city_race_aggregated.csv
1               RQ1  ...             city_race_aggregated.csv
2               RQ1  ...                        precision.csv
3               RQ1  ...  city_race_precision_200_to_500m.csv
4               RQ2  ...     city_race_orientation_events.csv
5               RQ2  ...           questionnaire_{method}.csv
6               RQ2  ...           questionnaire_{method}.csv
7               RQ3  ...      questionnaire_cybersickness.csv
8               RQ3  ...            questionnaire_raw-tlx.csv
9               RQ3  ...           questionnaire_{method}.csv
10              RQ3  ...        questionnaire_preferences.csv

[11 rows x 7 columns]


## 2. Load files and basic integrity checks


In [3]:
missing_files = [name for name in INPUT_FILES.values() if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing input files: {missing_files}')

raw = {key: pd.read_csv(DATA_DIR / filename) for key, filename in INPUT_FILES.items()}
required_input_file_count = len(INPUT_FILES)
loaded_input_file_count = len(raw)
print(f'Required input CSV files loaded: {loaded_input_file_count} / {required_input_file_count}')
print({key: frame.shape for key, frame in raw.items()})

def participant_uid(location, participant):
    p = participant.astype(str).str.replace(r'\.0$', '', regex=True).str.upper()
    p = p.where(p.str.startswith('P'), 'P' + p.str.zfill(2))
    return location.astype(str).str.upper() + '_' + p

coverage = {}
for key, frame in raw.items():
    pid = 'ParticipantID' if 'ParticipantID' in frame else 'Participant ID'
    if 'Location' in frame and pid in frame:
        coverage[key] = participant_uid(frame['Location'], frame[pid]).nunique()
display(pd.Series(coverage, name='Participants').to_frame())

# Audit City Race at its actual unit of observation. Missing routes do not
# remove a participant from other study datasets or from route-level LMMs.
city_audit = raw['city'][['Location', 'ParticipantID', 'Method', 'PathID']].copy()
city_audit['ParticipantUID'] = participant_uid(
    city_audit['Location'], city_audit['ParticipantID']
)
city_audit['Method'] = city_audit['Method'].astype(str).str.lower()
city_audit['PathID'] = city_audit['PathID'].astype(str).str.upper()
city_key_columns = ['ParticipantUID', 'Method', 'PathID']

city_participant_uids = sorted(city_audit['ParticipantUID'].unique())
expected_city_index = pd.MultiIndex.from_product(
    [city_participant_uids, METHODS, ROUTES], names=city_key_columns
)
observed_city_index = pd.MultiIndex.from_frame(
    city_audit[city_key_columns].drop_duplicates()
)
city_missing_observations = expected_city_index.difference(
    observed_city_index
).to_frame(index=False)
city_missing_observations['MissingObservation'] = True

city_observation_counts = city_audit.groupby('ParticipantUID').size()
city_method_counts = city_audit.groupby('ParticipantUID')['Method'].nunique()
city_routes_per_method = city_audit.groupby(
    ['ParticipantUID', 'Method']
)['PathID'].nunique().unstack(fill_value=0).reindex(columns=METHODS, fill_value=0)
complete_city_mask = (
    city_observation_counts.eq(9)
    & city_method_counts.eq(3)
    & city_routes_per_method.eq(3).all(axis=1)
)
complete_city_participant_uids = set(
    complete_city_mask.index[complete_city_mask]
)

observed_city_n = len(city_participant_uids)
complete_city_n = len(complete_city_participant_uids)
incomplete_city_n = observed_city_n - complete_city_n
observed_city_route_observations = len(observed_city_index)
expected_city_route_observations = observed_city_n * len(METHODS) * len(ROUTES)
missing_city_route_observations = len(city_missing_observations)
observed_precision_n = coverage.get('precision', 0)

print('\nCity Race data summary')
print(f'Participants observed: {observed_city_n}')
print(f'Participants with complete 9/9 routes: {complete_city_n}')
print(f'Participants with incomplete route data: {incomplete_city_n}')
print(
    'City Race route observations processed: '
    f'{observed_city_route_observations} / {expected_city_route_observations} expected'
)
if city_missing_observations.empty:
    print('Missing route observations: none')
else:
    warnings.warn(
        f'{missing_city_route_observations} City Race route observation(s) are missing. '
        'Participant-level route summaries require complete A/B/C coverage for '
        'each participant × method × outcome; '
        'route-level analyses retain all available observations.'
    )
    print('Missing route observations:')
    for row in city_missing_observations.itertuples(index=False):
        print(f'  {row.ParticipantUID} — {METHOD_LABELS[row.Method]} — Route {row.PathID}')

checks = pd.DataFrame([
    {'Check': 'Required input CSV files loaded', 'Observed': loaded_input_file_count,
     'Expected': required_input_file_count,
     'OK': loaded_input_file_count == required_input_file_count},
    {'Check': 'City Race participants', 'Observed': observed_city_n,
     'Expected': observed_city_n, 'OK': True},
    {'Check': 'Complete City Race participants (9/9)', 'Observed': complete_city_n,
     'Expected': observed_city_n, 'OK': complete_city_n == observed_city_n},
    {'Check': 'Incomplete City Race participants', 'Observed': incomplete_city_n,
     'Expected': 0, 'OK': incomplete_city_n == 0},
    {'Check': 'Observed City Race route observations',
     'Observed': observed_city_route_observations,
     'Expected': expected_city_route_observations,
     'OK': observed_city_route_observations == expected_city_route_observations},
    {'Check': 'Missing City Race route observations',
     'Observed': missing_city_route_observations, 'Expected': 0,
     'OK': missing_city_route_observations == 0},
    {'Check': 'Precision rows', 'Observed': len(raw['precision']),
     'Expected': observed_precision_n * 27,
     'OK': len(raw['precision']) == observed_precision_n * 27},
])
display(checks)

if (EXPECTED_CURRENT_SAMPLE_SIZE is not None and
        observed_city_n != EXPECTED_CURRENT_SAMPLE_SIZE):
    warnings.warn(
        f'City Race contains N={observed_city_n}, while '
        f'EXPECTED_CURRENT_SAMPLE_SIZE={EXPECTED_CURRENT_SAMPLE_SIZE}. '
        'Analysis will continue with the observed data.'
    )
non_city_failures = checks.loc[
    ~checks['OK'] & ~checks['Check'].str.contains('City Race', case=False)
]
if not non_city_failures.empty:
    warnings.warn(
        'One or more non-City-Race integrity checks failed. Analysis will '
        'continue, but inspect the table above before interpreting results.'
    )

SAMPLE_SIZE = observed_city_n
print(f'Overall observed City Race sample: N={SAMPLE_SIZE}.')
print('Participant-level paired N is contrast- and outcome-specific; see result tables.')


Required input CSV files loaded: 13 / 13
{'city': (378, 39), 'precision': (1134, 20), 'baseline': (42, 10), 'map': (42, 10), 'tsunami': (42, 10), 'csq': (42, 18), 'tlx': (42, 5), 'sbsod': (42, 4), 'demography': (42, 10), 'preferences': (42, 14), 'city_precision': (870, 34), 'orientation_events': (2231, 66), 'checkpoint_episodes': (1492, 59)}
                     Participants
city                           42
precision                      42
baseline                       42
map                            42
tsunami                        42
csq                            42
tlx                            42
sbsod                          42
demography                     42
preferences                    42
city_precision                 42
orientation_events             42
checkpoint_episodes            42

City Race data summary
Participants observed: 42
Participants with complete 9/9 routes: 42
Participants with incomplete route data: 0
City Race route observations processed: 378 /

## 3. Prepare analysis-ready tables


In [4]:
COUNTERBALANCE_SLOT_OVERRIDES = {
    19: 18, 20: 15, 21: 12, 22: 7, 23: 4, 24: 1,
}

def method_sequence(participant_id):
    number = int(float(str(participant_id).upper().replace('P', '')))
    design_slot = number if 1 <= number <= 18 else COUNTERBALANCE_SLOT_OVERRIDES.get(number)
    if design_slot is None:
        raise ValueError(
            f'No counterbalancing design slot is defined for {participant_id}.'
        )
    return METHOD_SEQUENCES[(design_slot - 1) // 3]

city = raw['city'].copy()
city.insert(0, 'ParticipantUID', participant_uid(city['Location'], city['ParticipantID']))
city['IdealRouteLength'] = city['PathID'].map(ROUTE_LENGTHS_M)
city['NetTimePer100m'] = city['NetPathTime'] / city['IdealRouteLength'] * 100
city['TotalTimePer100m'] = city['TotalPathTime'] / city['IdealRouteLength'] * 100
city['TraversedDistanceRatio'] = city['TotalTraversedDistance'] / city['IdealRouteLength']
city['PathEfficiency'] = city['IdealRouteLength'] / city['TotalTraversedDistance']

precision = raw['precision'].copy()
precision['Distance'] = pd.to_numeric(precision['Distance'], errors='raise').astype(int)
precision['TargetPos_x'] = precision['Distance'].map(PRECISION_TARGET_X)
precision['TargetPos_z'] = 0.0
if precision['TargetPos_x'].isna().any():
    unknown = sorted(precision.loc[precision['TargetPos_x'].isna(), 'Distance'].unique())
    raise ValueError(f'Unknown precision distance(s): {unknown}')
precision['LandingOffset_x'] = precision['LandingPos_x'] - precision['TargetPos_x']
precision['LandingOffset_z'] = precision['LandingPos_z'] - precision['TargetPos_z']
precision['DistanceLabel'] = precision['Distance'].map(PRECISION_DISTANCE_LABELS)

# Validate that the recorded Euclidean error and the reconstructed target-centred
# coordinates describe the same horizontal landing error (allowing CSV rounding).

# Trial exclusions are embedded in precision.csv by raw-log adjudication.
# Statistical outlier status alone never removes a trial.
decision_keys = ['ParticipantUID', 'Method', 'Distance', 'Trial', 'SourceFile']
required_decision_columns = {
    'RecommendedExclude', 'RepeatedFailureGroup',
    'RepeatedFailureTrialExclude', 'DecisionReason',
}
missing_decision_columns = required_decision_columns - set(precision.columns)
if missing_decision_columns:
    raise ValueError(
        'precision.csv lacks embedded adjudication columns. Regenerate it with '
        'the current precision preprocessing pipeline. Missing columns: '
        f'{sorted(missing_decision_columns)}'
    )

def strict_boolean_flag(frame, column):
    normalized = frame[column].astype(str).str.strip().str.lower()
    mapping = {
        'true': True, 'false': False, '1': True, '0': False,
        'yes': True, 'no': False,
    }
    unknown = sorted(set(normalized.dropna()) - set(mapping))
    if unknown:
        raise ValueError(f'{column} contains invalid Boolean values: {unknown}')
    return normalized.map(mapping).astype(bool)

for flag_column in (
    'RecommendedExclude', 'RepeatedFailureGroup',
    'RepeatedFailureTrialExclude',
):
    precision[flag_column] = strict_boolean_flag(precision, flag_column)

precision['ExcludeFailedManipulation'] = (
    precision['RecommendedExclude']
    | precision['RepeatedFailureTrialExclude']
)
precision['DecisionReason'] = precision['DecisionReason'].fillna('Retained')
precision_full = precision.copy()
precision_clean = precision.loc[~precision['ExcludeFailedManipulation']].copy()
print(
    f'Precision adjudication: {len(precision_clean)} retained, '
    f'{precision["ExcludeFailedManipulation"].sum()} excluded.'
)

city_precision = raw['city_precision'].copy()
if 'MethodOrder' not in city_precision.columns:
    city_precision['MethodOrder'] = pd.to_numeric(
        city_precision['Run'], errors='raise'
    ).map(lambda run: 1 if 1 <= run <= 3 else 2 if 5 <= run <= 7 else 3)
numeric_city_precision = [
    'MethodOrder', 'Run', 'TeleportSourceRow', 'AppearSourceRow',
    'ProgressBannerSourceRow', 'StartPos_x', 'StartPos_y', 'StartPos_z',
    'TargetPos_x', 'TargetPos_y', 'TargetPos_z', 'LandingPos_x', 'LandingPos_y',
    'LandingPos_z', 'LandingOffset_x', 'LandingOffset_y', 'LandingOffset_z',
    'TargetDistance2D', 'TargetDistance3D', 'ActualDisplacement2D',
    'ActualDisplacement3D', 'LandingError2D', 'LandingError3D',
]
for column in numeric_city_precision:
    city_precision[column] = pd.to_numeric(city_precision[column], errors='raise')
city_precision['TargetID'] = (
    city_precision['Route'].astype(str) + '_' +
    city_precision['TargetPos_x'].round(1).astype(str) + '_' +
    city_precision['TargetPos_z'].round(1).astype(str)
)
city_precision['Log1pLandingError2D'] = np.log1p(city_precision['LandingError2D'])
CITY_PRECISION_BANDS = ('200-350', '350-500')
city_precision['DistanceBandAnalysis'] = pd.cut(
    city_precision['TargetDistance2D'], bins=[200, 350, 500],
    right=False, labels=CITY_PRECISION_BANDS,
)
city_precision['DistanceBandAnalysis'] = pd.Categorical(
    city_precision['DistanceBandAnalysis'],
    categories=CITY_PRECISION_BANDS, ordered=True,
)

checkpoint_episodes = raw['checkpoint_episodes'].copy()
numeric_checkpoint_episode_columns = [
    'MethodOrder', 'Run', 'CheckpointIndex', 'FirstAimSourceRow',
    'FirstTeleportSourceRow', 'FirstAppearSourceRow', 'FinalAppearSourceRow',
    'ProgressBannerSourceRow', 'StartPos_x', 'StartPos_y', 'StartPos_z',
    'TargetPos_x', 'TargetPos_y', 'TargetPos_z',
    'FirstLandingPos_x', 'FirstLandingPos_y', 'FirstLandingPos_z',
    'FinalLandingPos_x', 'FinalLandingPos_y', 'FinalLandingPos_z',
    'InitialTargetDistance2D', 'InitialTargetDistance3D',
    'FirstTeleportDistance2D', 'FirstTeleportDistance3D',
    'FirstLandingError2D', 'FirstLandingError3D',
    'FinalLandingError2D', 'FinalLandingError3D',
    'CorrectionCount', 'TeleportCountToCheckpoint',
    'TotalCorrectionDistance2D', 'TotalCorrectionDistance3D',
    'MaxCorrectionDistance2D', 'MaxCorrectionDistance3D',
    'FirstCorrectionLatency', 'TotalEpisodeTime', 'BannerDelayTime',
]
for column in numeric_checkpoint_episode_columns:
    checkpoint_episodes[column] = pd.to_numeric(
        checkpoint_episodes[column], errors='coerce'
    )
for column in ('DirectHit', 'AllCorrectionsUnder100m',
               'ContainsCorrectionAtLeast100m'):
    checkpoint_episodes[column] = (
        checkpoint_episodes[column].astype(str).str.lower().map(
            {'true': True, 'false': False}
        )
    )
checkpoint_episodes['DistanceBandAnalysis'] = pd.cut(
    checkpoint_episodes['InitialTargetDistance2D'], bins=[200, 350, 500],
    right=False, labels=CITY_PRECISION_BANDS,
)
checkpoint_episodes['DistanceBandAnalysis'] = pd.Categorical(
    checkpoint_episodes['DistanceBandAnalysis'],
    categories=CITY_PRECISION_BANDS, ordered=True,
)
checkpoint_episodes['TargetID'] = (
    checkpoint_episodes['Route'].astype(str) + '_' +
    checkpoint_episodes['TargetPos_x'].round(1).astype(str) + '_' +
    checkpoint_episodes['TargetPos_z'].round(1).astype(str)
)
checkpoint_episodes['Log1pFirstLandingError2D'] = np.log1p(
    checkpoint_episodes['FirstLandingError2D']
)
checkpoint_episodes['Distance100Centered'] = (
    checkpoint_episodes['InitialTargetDistance2D']
    - checkpoint_episodes['InitialTargetDistance2D'].mean()
) / 100
city_precision['Distance100Centered'] = (
    city_precision['TargetDistance2D'] - city_precision['TargetDistance2D'].mean()
) / 100

orientation_events = raw['orientation_events'].copy()
post_confirmation_required_columns = {
    'NextAimSourceRow', 'NextAimTimestamp',
    'NextAimHeadVec_x', 'NextAimHeadVec_y', 'NextAimHeadVec_z',
    'ProgressBannerTimestamp', 'ProgressBannerHeadVec_x',
    'ProgressBannerHeadVec_y', 'ProgressBannerHeadVec_z',
}
POST_CONFIRMATION_COLUMNS_PRESENT = (
    post_confirmation_required_columns.issubset(orientation_events.columns)
)
numeric_orientation_columns = [
    'MethodOrder', 'Run', 'CheckpointIndex', 'TeleportSourceRow',
    'AppearSourceRow', 'ProgressBannerSourceRow', 'NextAimSourceRow',
    'BannerDelayTime',
    'AppearPos_x', 'AppearPos_y', 'AppearPos_z',
    'ProgressBannerPos_x', 'ProgressBannerPos_y', 'ProgressBannerPos_z',
    'AppearHeadVec_x', 'AppearHeadVec_y', 'AppearHeadVec_z',
    'ProgressBannerHeadVec_x', 'ProgressBannerHeadVec_y',
    'ProgressBannerHeadVec_z', 'HeadRotation3D', 'HeadRotationXZ',
    'AppearHeadYaw', 'ProgressBannerHeadYaw', 'SignedYawChange',
    'AbsoluteYawChange', 'AppearHeadPitch', 'ProgressBannerHeadPitch',
    'SignedPitchChange', 'AbsolutePitchChange',
    'AppearHeadToLeftControllerAngle3D',
    'AppearHeadToRightControllerAngle3D',
    'ProgressBannerHeadToLeftControllerAngle3D',
    'ProgressBannerHeadToRightControllerAngle3D',
    'NextAimHeadVec_x', 'NextAimHeadVec_y', 'NextAimHeadVec_z',
    'PostBannerAimLatency',
    'HeadRotationBannerToNextAim3D', 'HeadRotationBannerToNextAimXZ',
    'ReachedTargetPos_x', 'ReachedTargetPos_y', 'ReachedTargetPos_z',
    'NextTargetPos_x', 'NextTargetPos_y', 'NextTargetPos_z',
    'IncomingTeleportDistance2D', 'IncomingTeleportDistance3D',
    'NextTargetDistanceAtAppear2D', 'NextTargetDistanceAtAppear3D',
    'DirectionalErrorAtAppear3D', 'DirectionalErrorAtProgressBanner3D',
    'DirectionalErrorReduction3D',
]
for column in numeric_orientation_columns:
    if column not in orientation_events.columns:
        # Some processed inputs may not contain the optional next-aim fields:
        # confirmatory H2.1 remains available and supportive RC is skipped below.
        orientation_events[column] = np.nan
    orientation_events[column] = pd.to_numeric(
        orientation_events[column], errors='raise'
    )
POST_CONFIRMATION_AIMING_AVAILABLE = (
    POST_CONFIRMATION_COLUMNS_PRESENT
    and orientation_events['NextAimHeadVec_x'].notna().any()
)
if POST_CONFIRMATION_AIMING_AVAILABLE:
    banner_time = pd.to_datetime(
        orientation_events['ProgressBannerTimestamp'], errors='coerce'
    )
    next_aim_time = pd.to_datetime(
        orientation_events['NextAimTimestamp'], errors='coerce'
    )
    orientation_events['PostBannerAimLatency'] = (
        next_aim_time - banner_time
    ).dt.total_seconds()
    banner_vectors = orientation_events[[
        'ProgressBannerHeadVec_x', 'ProgressBannerHeadVec_y',
        'ProgressBannerHeadVec_z'
    ]].to_numpy(dtype=float)
    aim_vectors = orientation_events[[
        'NextAimHeadVec_x', 'NextAimHeadVec_y', 'NextAimHeadVec_z'
    ]].to_numpy(dtype=float)
    vector_denominator = (
        np.linalg.norm(banner_vectors, axis=1)
        * np.linalg.norm(aim_vectors, axis=1)
    )
    vector_cosine = np.divide(
        np.sum(banner_vectors * aim_vectors, axis=1), vector_denominator,
        out=np.full(len(orientation_events), np.nan),
        where=vector_denominator > 0,
    )
    orientation_events['HeadRotationBannerToNextAim3D'] = np.degrees(
        np.arccos(np.clip(vector_cosine, -1, 1))
    )


# The current preprocessing already defines ecological landing coordinates from
# the checkpoint-confirming progress_banner event. Verify that contract without
# rewriting analysis values (CSV-derived errors may differ by rounding only).
checkpoint_event_keys = [
    'ParticipantUID', 'SourceFile', 'ProgressBannerSourceRow',
]
checkpoint_positions = orientation_events[
    checkpoint_event_keys + [
        'ProgressBannerPos_x', 'ProgressBannerPos_y', 'ProgressBannerPos_z',
    ]
].drop_duplicates(checkpoint_event_keys)

city_precision_confirmation = city_precision.merge(
    checkpoint_positions, on=checkpoint_event_keys, how='left',
    validate='one_to_one', suffixes=('', '_confirmation'),
)
if city_precision_confirmation['ProgressBannerPos_x'].isna().any():
    raise ValueError('Missing progress_banner position for City Race precision.')
for axis in ('x', 'y', 'z'):
    if not np.allclose(
        city_precision_confirmation[f'LandingPos_{axis}'],
        city_precision_confirmation[f'ProgressBannerPos_{axis}'],
        rtol=0, atol=1e-9,
    ):
        raise ValueError(
            'City Race precision landing positions are not based on '
            'checkpoint-confirming progress_banner events.'
        )
for dimension, axes in (('2D', ('x', 'z')), ('3D', ('x', 'y', 'z'))):
    reconstructed_error = np.sqrt(sum(
        (city_precision_confirmation[f'ProgressBannerPos_{axis}']
         - city_precision_confirmation[f'TargetPos_{axis}']) ** 2
        for axis in axes
    ))
    if not np.allclose(
        city_precision_confirmation[f'LandingError{dimension}'],
        reconstructed_error, rtol=1e-9, atol=1e-6,
    ):
        raise ValueError(
            f'City Race precision LandingError{dimension} is inconsistent '
            'with progress_banner positions.'
        )

checkpoint_episode_confirmation = checkpoint_episodes.merge(
    checkpoint_positions, on=checkpoint_event_keys, how='left',
    validate='many_to_one', suffixes=('', '_confirmation'),
)
if checkpoint_episode_confirmation['ProgressBannerPos_x'].isna().any():
    raise ValueError('Missing progress_banner position for a checkpoint episode.')
for axis in ('x', 'y', 'z'):
    if not np.allclose(
        checkpoint_episode_confirmation[f'FinalLandingPos_{axis}'],
        checkpoint_episode_confirmation[f'ProgressBannerPos_{axis}'],
        rtol=0, atol=1e-9,
    ):
        raise ValueError(
            'Checkpoint-episode final positions are not based on '
            'checkpoint-confirming progress_banner events.'
        )
print('City Race landing-position semantics verified against progress_banner.')

orientation_events['TargetID'] = (
    orientation_events['Route'].astype(str) + '_' +
    orientation_events['CheckpointIndex'].astype(int).astype(str)
)
orientation_events['Log1pDirectionalErrorAtAppear3D'] = np.log1p(
    orientation_events['DirectionalErrorAtAppear3D']
)
orientation_events['Log1pHeadRotation3D'] = np.log1p(
    orientation_events['HeadRotation3D']
)
orientation_events['Log1pBannerDelayTime'] = np.log1p(
    orientation_events['BannerDelayTime']
)
dominant_hand = raw['demography'][
    ['Location', 'Participant ID', 'Dominant hand']
].copy()
dominant_hand['ParticipantUID'] = participant_uid(
    dominant_hand['Location'], dominant_hand['Participant ID']
)
dominant_hand = dominant_hand[['ParticipantUID', 'Dominant hand']].rename(
    columns={'Dominant hand': 'DominantHand'}
)
orientation_events = orientation_events.merge(
    dominant_hand, on='ParticipantUID', how='left', validate='many_to_one'
)
if orientation_events['DominantHand'].isna().any():
    raise ValueError('Missing dominant-hand information for orientation events.')
right_dominant = orientation_events['DominantHand'].str.lower().eq('right')
orientation_events['NonDominantHand'] = np.where(
    right_dominant, 'left', 'right'
)
orientation_events['AppearHeadToNonDominantControllerAngle3D'] = np.where(
    right_dominant,
    orientation_events['AppearHeadToLeftControllerAngle3D'],
    orientation_events['AppearHeadToRightControllerAngle3D'],
)
orientation_events['ProgressBannerHeadToNonDominantControllerAngle3D'] = np.where(
    right_dominant,
    orientation_events['ProgressBannerHeadToLeftControllerAngle3D'],
    orientation_events['ProgressBannerHeadToRightControllerAngle3D'],
)
orientation_events['NonDominantControllerAttentionShift'] = (
    orientation_events['ProgressBannerHeadToNonDominantControllerAngle3D'] -
    orientation_events['AppearHeadToNonDominantControllerAngle3D']
)
for threshold in (15, 20, 25):
    orientation_events[f'LookingAtNonDominantController{threshold}'] = (
        orientation_events['AppearHeadToNonDominantControllerAngle3D'] <= threshold
    )
if (orientation_events['BannerDelayTime'] < 0).any():
    raise ValueError('Orientation events contain a negative banner delay.')
if orientation_events.duplicated([
    'ParticipantUID', 'SourceFile', 'ProgressBannerSourceRow'
]).any():
    raise ValueError('Orientation event keys are not unique.')
orientation_method_coverage = orientation_events.groupby(
    'ParticipantUID')['Method'].nunique()
if not orientation_method_coverage.eq(3).all():
    raise ValueError('Every included participant must have orientation events for all methods.')

SPATIAL_COLUMNS = {
    'position_awareness': 'After teleporting, I remained aware of my position in the environment.',
    'direction_awareness': 'After teleporting, I remained aware of the direction I was facing.',
    'reorientation_effort': 'After teleporting, I needed additional effort to reorient myself.',
    'environmental_continuity': 'Movement between locations felt spatially continuous.',
    'environment_distance': 'I was able to accurately estimate distances between locations while navigating.',
    'destination_distance': 'I was able to accurately estimate the distance to my intended teleport destination.',
    'easy_to_learn': 'I found this locomotion technique easy to learn.',
    'intuitive': 'I found this locomotion technique intuitive to use.',
}

spatial_parts = []
for method in METHODS:
    frame = raw[method]
    out = pd.DataFrame({
        'ParticipantUID': participant_uid(frame['Location'], frame['Participant ID']),
        'ParticipantID': frame['Participant ID'], 'Location': frame['Location'], 'Method': method,
    })
    for short, column in SPATIAL_COLUMNS.items():
        out[short] = pd.to_numeric(frame[column], errors='coerce')
    out['reorientation_effort_reversed'] = 7 - out['reorientation_effort']
    out['SpatialAwarenessComposite'] = out[
        ['position_awareness', 'direction_awareness', 'reorientation_effort_reversed']
    ].mean(axis=1)
    out['DistanceAwarenessComposite'] = out[['environment_distance', 'destination_distance']].mean(axis=1)
    out['UsabilityComposite'] = out[['easy_to_learn', 'intuitive']].mean(axis=1)
    spatial_parts.append(out)
spatial = pd.concat(spatial_parts, ignore_index=True)

tlx = raw['tlx'].copy()
tlx['ParticipantUID'] = participant_uid(tlx['Location'], tlx['Participant ID'])
tlx_long = tlx.melt(
    id_vars=['ParticipantUID', 'Participant ID', 'Location'],
    value_vars=['Baseline TLX', 'Minimap TLX', 'Tsunami TLX'],
    var_name='MethodLabel', value_name='RAWTLX')
tlx_long['Method'] = tlx_long['MethodLabel'].map(
    {'Baseline TLX': 'baseline', 'Minimap TLX': 'map', 'Tsunami TLX': 'tsunami'})

csq_rows = []
csq_time_rows = []
for _, row in raw['csq'].iterrows():
    uid = participant_uid(pd.Series([row['Location']]), pd.Series([row['Participant ID']])).iloc[0]
    for measurement in range(4):
        csq_time_rows.append({
            'ParticipantUID': uid,
            'ParticipantID': row['Participant ID'],
            'Location': row['Location'],
            'Measurement': measurement,
            'MeasurementLabel': (
                'Pre-experiment' if measurement == 0
                else f'After method block {measurement}'
            ),
            'CSQTotal': float(row[f'CSQ-{measurement} Total']),
            'CSQNausea': float(row[f'CSQ-{measurement} Nausea']),
            'CSQVestibular': float(row[f'CSQ-{measurement} Vestibular']),
            'CSQOculomotor': float(row[f'CSQ-{measurement} Oculomotor']),
        })
    previous = float(row['CSQ-0 Total'])
    for session, method in enumerate(method_sequence(row['Participant ID']), start=1):
        total = float(row[f'CSQ-{session} Total'])
        csq_rows.append({
            'ParticipantUID': uid, 'ParticipantID': row['Participant ID'],
            'Location': row['Location'], 'MethodOrder': session, 'Method': method,
            'CSQTotalBefore': previous, 'CSQTotalAfter': total,
            'CSQTotalChange': total - previous,
        })
        previous = total
csq_long = pd.DataFrame(csq_rows)
csq_time = pd.DataFrame(csq_time_rows)

print('Prepared:', {'city': city.shape, 'precision_full': precision_full.shape,
                    'precision_clean': precision_clean.shape,
                    'city_precision': city_precision.shape,
                    'orientation_events': orientation_events.shape,
                    'spatial': spatial.shape,
                    'tlx': tlx_long.shape, 'csq': csq_long.shape})


Precision adjudication: 1115 retained, 19 excluded.
City Race landing-position semantics verified against progress_banner.
Prepared: {'city': (378, 45), 'precision_full': (1134, 26), 'precision_clean': (1115, 26), 'city_precision': (870, 38), 'orientation_events': (2231, 78), 'spatial': (126, 16), 'tlx': (126, 6), 'csq': (126, 8)}


Tsunami_quantitative_analysis.ipynb:code-cell-4:212: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
Tsunami_quantitative_analysis.ipynb:code-cell-4:215: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


## 4. Statistical utilities


In [5]:
_AGGREGATE_CACHE = {}
_ROUTE_COVERAGE_AUDIT = {}

def aggregate_method(frame, outcome):
    # Participant × method summaries of route-derived outcomes are eligible
    # only when that outcome has at least one valid observation on A, B, and C.
    # Event-level weighting within an eligible group remains unchanged.
    cache_key = (id(frame), outcome)
    cached = _AGGREGATE_CACHE.get(cache_key)
    if cached is not None and cached[0] is frame:
        return cached[1].copy()

    columns = ['ParticipantUID', 'Method', outcome]
    route_derived = 'Route' in frame.columns or 'PathID' in frame.columns
    if not route_derived:
        result = frame.groupby(['ParticipantUID', 'Method'], as_index=False)[outcome].mean()
        _AGGREGATE_CACHE[cache_key] = (frame, result)
        return result.copy()

    route_column = 'Route' if 'Route' in frame.columns else 'PathID'
    data = frame[columns + [route_column]].copy()
    data[route_column] = data[route_column].astype(str).str.upper()
    valid = data.loc[data[outcome].notna() & data[route_column].isin(ROUTES)]
    coverage = valid.groupby(['ParticipantUID', 'Method'])[route_column].agg(
        lambda values: tuple(sorted(set(values)))
    )
    all_combinations = data[['ParticipantUID', 'Method']].drop_duplicates()
    coverage_table = all_combinations.merge(
        coverage.rename('RoutesTuple').reset_index(),
        on=['ParticipantUID', 'Method'], how='left',
    )
    coverage_table['RoutesTuple'] = coverage_table['RoutesTuple'].apply(
        lambda value: value if isinstance(value, tuple) else tuple()
    )
    expected_routes = set(ROUTES)
    coverage_table['CompleteRouteCoverage'] = coverage_table['RoutesTuple'].apply(
        lambda value: set(value) == expected_routes
    )
    incomplete = coverage_table.loc[~coverage_table['CompleteRouteCoverage']].copy()
    for row in incomplete.itertuples(index=False):
        present = set(row.RoutesTuple)
        audit_key = (row.ParticipantUID, row.Method, outcome)
        _ROUTE_COVERAGE_AUDIT[audit_key] = {
            'ParticipantUID': row.ParticipantUID,
            'Method': row.Method,
            'Outcome': outcome,
            'RoutesPresent': '|'.join(route for route in ROUTES if route in present),
            'RoutesMissing': '|'.join(route for route in ROUTES if route not in present),
            'AnalysisLevel': 'participant × method route aggregation',
        }
    eligible = coverage_table.loc[
        coverage_table['CompleteRouteCoverage'], ['ParticipantUID', 'Method']
    ]
    result = (
        valid.merge(eligible, on=['ParticipantUID', 'Method'], how='inner')
        .groupby(['ParticipantUID', 'Method'], as_index=False)[outcome].mean()
    )
    _AGGREGATE_CACHE[cache_key] = (frame, result)
    return result.copy()

def paired_mean_ci(diff):
    diff = np.asarray(diff, dtype=float)
    return tuple(stats.t.interval(CONFIDENCE_LEVEL, len(diff)-1,
                                  loc=diff.mean(), scale=stats.sem(diff)))

def paired_rank_biserial(diff):
    # Signed matched-pairs rank-biserial correlation (first minus second).
    values = np.asarray(diff, dtype=float)
    values = values[~np.isclose(values, 0)]
    if len(values) == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(values), method='average')
    denominator = ranks.sum()
    return float((ranks[values > 0].sum() - ranks[values < 0].sum()) / denominator)

def paired_rank_biserial_bootstrap(samples):
    # Vectorized row-wise matched-pairs rank-biserial correlations. This is
    # equivalent to paired_rank_biserial applied row by row, including ties and
    # zero handling, and avoids repeated Python-level scipy calls.
    samples = np.asarray(samples, dtype=float)
    nonzero = ~np.isclose(samples, 0)
    ranks = stats.rankdata(
        np.where(nonzero, np.abs(samples), np.nan), axis=1,
        method='average', nan_policy='omit',
    )
    denominator = np.nansum(ranks, axis=1)
    numerator = np.nansum(
        np.where(samples > 0, ranks, np.where(samples < 0, -ranks, 0.0)),
        axis=1,
    )
    return np.divide(
        numerator, denominator,
        out=np.zeros(samples.shape[0], dtype=float), where=denominator > 0,
    )

def hodges_lehmann_paired(diff):
    # One-sample Hodges–Lehmann estimator from Walsh averages.
    values = np.asarray(diff, dtype=float)
    upper = np.triu_indices(len(values))
    return float(np.median((values[:, None] + values[None, :])[upper] / 2))

_PAIRED_EFFECT_CACHE = {}

def paired_effect_summary(diff, outcome):
    # Parametric and rank-based effect summaries with reproducible bootstrap CIs.
    values = np.asarray(diff, dtype=float)
    cache_key = (outcome, values.shape, values.tobytes())
    cached = _PAIRED_EFFECT_CACHE.get(cache_key)
    if cached is not None:
        return cached.copy()
    sd = values.std(ddof=1)
    dz = values.mean() / sd if sd > 0 else np.nan
    hl = hodges_lehmann_paired(values)
    rrb = paired_rank_biserial(values)
    seed = RANDOM_SEED ^ zlib.crc32(values.tobytes()) ^ zlib.crc32(outcome.encode())
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(values), size=(EFFECT_BOOTSTRAP_RESAMPLES, len(values)))
    samples = values[indices]
    sample_sd = samples.std(axis=1, ddof=1)
    sample_dz = np.divide(
        samples.mean(axis=1), sample_sd,
        out=np.full(EFFECT_BOOTSTRAP_RESAMPLES, np.nan), where=sample_sd > 0,
    )
    upper = np.triu_indices(len(values))
    sample_hl = np.median(
        (samples[:, :, None] + samples[:, None, :])[:, upper[0], upper[1]] / 2,
        axis=1,
    )
    sample_rrb = paired_rank_biserial_bootstrap(samples)
    quantiles = [(1-CONFIDENCE_LEVEL)/2, 1-(1-CONFIDENCE_LEVEL)/2]
    threshold = PRACTICAL_THRESHOLDS.get(outcome)
    mean_low, mean_high = paired_mean_ci(values)
    practically_distinct = (
        bool(mean_low > threshold or mean_high < -threshold)
        if threshold is not None else np.nan
    )
    result = {
        'EffectDz': dz,
        'EffectDzCI95Low': np.nanquantile(sample_dz, quantiles[0]),
        'EffectDzCI95High': np.nanquantile(sample_dz, quantiles[1]),
        'RankBiserial': rrb,
        'RankBiserialCI95Low': np.quantile(sample_rrb, quantiles[0]),
        'RankBiserialCI95High': np.quantile(sample_rrb, quantiles[1]),
        'HodgesLehmann': hl,
        'HLCI95Low': np.quantile(sample_hl, quantiles[0]),
        'HLCI95High': np.quantile(sample_hl, quantiles[1]),
        'PracticalThreshold': threshold,
        'CIExcludesPracticalNull': practically_distinct,
    }
    _PAIRED_EFFECT_CACHE[cache_key] = result.copy()
    return result

def holm(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values))
    running = 0.0
    for rank, index in enumerate(order):
        running = max(running, (len(p_values)-rank) * p_values[index])
        adjusted[index] = min(1.0, running)
    return adjusted

RESULTS = {}
DIFFERENCES = {}
PAIRED_COMPARISONS = []

def register_paired_comparison(
    family, analysis, outcome, first, second, alternative, diff
):
    key = (family, analysis, outcome, first, second, alternative)
    existing = next(
        (row for row in PAIRED_COMPARISONS if row['RegistryKey'] == key), None
    )
    if existing is not None:
        return existing
    registered = {
        'RegistryKey': key, 'Hypothesis': family, 'Analysis': analysis,
        'Metric': outcome, 'First': first, 'Second': second,
        'Contrast': f'{METHOD_LABELS[first]} - {METHOD_LABELS[second]}',
        'Alternative': alternative, 'Differences': np.asarray(diff, dtype=float),
    }
    PAIRED_COMPARISONS.append(registered)
    return registered

def difference_diagnostics(diff):
    diff = np.asarray(diff, dtype=float)
    shapiro = stats.shapiro(diff) if len(diff) >= 3 and not np.allclose(diff, diff[0]) else None
    shapiro_w = float(shapiro.statistic) if shapiro is not None else np.nan
    shapiro_p = float(shapiro.pvalue) if shapiro is not None else np.nan
    ordered = np.sort(diff)
    theoretical = stats.norm.ppf((np.arange(1, len(diff) + 1) - 0.375) / (len(diff) + 0.25))
    qq_correlation = float(np.corrcoef(ordered, theoretical)[0, 1]) if len(diff) >= 3 else np.nan
    skewness = float(stats.skew(diff, bias=False)) if len(diff) >= 3 else np.nan
    sd = diff.std(ddof=1) if len(diff) >= 2 else np.nan
    max_abs_z = float(np.max(np.abs((diff - diff.mean()) / sd))) if np.isfinite(sd) and sd > 0 else 0.0
    q1, q3 = np.quantile(diff, [0.25, 0.75])
    iqr = q3 - q1
    tukey_outliers = int(np.sum((diff < q1 - 1.5 * iqr) | (diff > q3 + 1.5 * iqr)))
    approximately_normal = bool(
        np.isfinite(shapiro_p) and shapiro_p >= ALPHA
        and qq_correlation >= 0.95 and abs(skewness) <= 1.0 and max_abs_z < 3.0
    )
    return {
        'ShapiroW': shapiro_w, 'ShapiroP': shapiro_p,
        'QQCorrelation': qq_correlation, 'Skewness': skewness,
        'MaxAbsZ': max_abs_z, 'TukeyOutliers': tukey_outliers,
        'ApproximatelyNormal': approximately_normal,
        'RecommendedConventionalTest': (
            'Paired t-test' if approximately_normal else 'Wilcoxon signed-rank test'
        ),
    }

def residual_diagnostic_plot(residuals_by_outcome, title):
    fig, axes = plt.subplots(1, len(residuals_by_outcome),
                             figsize=(6 * len(residuals_by_outcome), 4.5), squeeze=False)
    rows = []
    for ax, (outcome, residuals) in zip(axes[0], residuals_by_outcome.items()):
        diagnostics = difference_diagnostics(np.asarray(residuals, dtype=float))
        stats.probplot(residuals, dist='norm', plot=ax)
        ax.set_title(outcome)
        ax.grid(alpha=0.25)
        rows.append({'Outcome': outcome, **diagnostics})
        residual_normality_plausible = bool(
            diagnostics['ShapiroP'] >= ALPHA
            and diagnostics['QQCorrelation'] >= 0.98
            and abs(diagnostics['Skewness']) <= 1.0
        )
        conclusion = 'approximately normal residuals' if residual_normality_plausible else 'residual-normality concerns'
        rows[-1]['ResidualNormalityPlausible'] = residual_normality_plausible
        print(
            f"{outcome}: Shapiro–Wilk p={diagnostics['ShapiroP']:.4f}, "
            f"Q–Q r={diagnostics['QQCorrelation']:.3f}, skew={diagnostics['Skewness']:.3f}; "
            f"{conclusion}. Interpret the transformed LMM together with this diagnostic."
        )
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    plt.show()
    table = pd.DataFrame(rows)
    display(table.style.format(precision=4))
    return table

def paired_wilcoxon_result(frame, outcome, first, second, alternative, family, analysis):
    # Pre-specified rank-based paired comparison, used for ordinal outcomes.
    wide = frame.pivot(index='ParticipantUID', columns='Method', values=outcome).dropna(subset=[first, second])
    first_values = wide[first].to_numpy(float)
    second_values = wide[second].to_numpy(float)
    diff = first_values - second_values
    low, high = paired_mean_ci(diff)
    label = f'{METHOD_LABELS[first]} - {METHOD_LABELS[second]}'
    if np.allclose(diff, 0):
        statistic, p_value = 0.0, 1.0
    else:
        test = stats.wilcoxon(
            first_values, second_values, alternative=alternative,
            zero_method='wilcox', method='auto',
        )
        statistic, p_value = float(test.statistic), float(test.pvalue)
    effects = paired_effect_summary(diff, outcome)
    registered = register_paired_comparison(
        family, analysis, outcome, first, second, alternative, diff
    )
    registered['CurrentSelectedMethod'] = 'Wilcoxon signed-rank test'
    registered['CurrentSelectedPRaw'] = p_value
    DIFFERENCES[(family, analysis, outcome, label)] = diff
    return {
        'Hypothesis': family, 'Analysis': analysis, 'Outcome': outcome,
        'Contrast': label, 'Alternative': alternative, 'N': len(diff),
        'MeanFirst': first_values.mean(), 'MeanSecond': second_values.mean(),
        'MeanDifference': diff.mean(), 'MedianDifference': np.median(diff),
        'CI95Low': low, 'CI95High': high, **effects,
        'TestStatistic': statistic, 'PValue': p_value,
        'TestMethod': 'Wilcoxon signed-rank test (pre-specified ordinal outcome)',
    }

def paired_primary_result(frame, outcome, first, second, alternative, family, analysis):
    # Publication pipeline: one pre-specified Wilcoxon procedure for every
    # simple participant-level paired comparison.
    wide = frame.pivot(index='ParticipantUID', columns='Method', values=outcome).dropna(subset=[first, second])
    first_values = wide[first].to_numpy(float)
    second_values = wide[second].to_numpy(float)
    diff = first_values - second_values
    low, high = paired_mean_ci(diff)
    label = f'{METHOD_LABELS[first]} - {METHOD_LABELS[second]}'
    if np.allclose(diff, 0):
        statistic, p_value = 0.0, 1.0
    else:
        test = stats.wilcoxon(
            diff, alternative=alternative, zero_method='wilcox',
            correction=False, method='auto',
        )
        statistic, p_value = float(test.statistic), float(test.pvalue)
    effects = paired_effect_summary(diff, outcome)
    registered = register_paired_comparison(
        family, analysis, outcome, first, second, alternative, diff
    )
    registered['CurrentSelectedMethod'] = 'Wilcoxon signed-rank test'
    registered['CurrentSelectedPRaw'] = p_value
    DIFFERENCES[(family, analysis, outcome, label)] = diff
    return {
        'Hypothesis': family, 'Analysis': analysis, 'Outcome': outcome,
        'Contrast': label, 'Alternative': alternative, 'N': len(diff),
        'NonzeroPairs': int(np.sum(~np.isclose(diff, 0))),
        'ZeroDifferences': int(np.sum(np.isclose(diff, 0))),
        'WilcoxonZeroMethod': 'wilcox', 'WilcoxonCorrection': False,
        'WilcoxonMethod': 'auto',
        'MeanFirst': first_values.mean(), 'MeanSecond': second_values.mean(),
        'MeanDifference': diff.mean(), 'MedianDifference': np.median(diff),
        'CI95Low': low, 'CI95High': high, **effects,
        'TestStatistic': statistic, 'PValue': p_value,
        'TestMethod': 'Wilcoxon signed-rank test',
    }

def show_results(key, rows, adjust=False):
    table = pd.DataFrame(rows)
    table['PHolm'] = holm(table['PValue']) if adjust else table['PValue']
    RESULTS[key] = table
    display(table.style.format(precision=4))
    return table

def method_boxplot(frame, outcome, title, ylabel):
    plot_data = aggregate_method(frame, outcome)
    order = list(METHODS)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    sns.boxplot(
        data=plot_data, x='Method', y=outcome, order=order,
        hue='Method', hue_order=order, palette=METHOD_COLORS,
        dodge=False, showfliers=False, width=0.6, ax=ax,
    )
    if ax.get_legend() is not None:
        ax.get_legend().remove()
    sns.stripplot(
        data=plot_data, x='Method', y=outcome, order=order,
        color='black', alpha=0.55, jitter=0.16, size=4, ax=ax,
    )
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([METHOD_LABELS[m] for m in order])
    ax.set_xlabel('Method'); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    plt.show()

def diverging_likert_plot(frame, items, title):
    categories = [1, 2, 3, 4, 5, 6]
    colors = {
        1: '#b2182b', 2: '#ef8a62', 3: '#fddbc7',
        4: '#d1e5f0', 5: '#67a9cf', 6: '#2166ac',
    }
    rows = []
    for method in METHODS:
        method_data = frame.loc[frame['Method'].eq(method)]
        for column, label in items:
            proportions = method_data[column].value_counts(normalize=True).reindex(categories, fill_value=0) * 100
            rows.append((f'{METHOD_LABELS[method]} — {label}', proportions))

    items_per_method = len(items)
    group_stride = items_per_method + 1
    y_positions = np.array([
        method * group_stride + item
        for method in range(len(METHODS))
        for item in range(items_per_method)
    ], dtype=float)
    plot_rows = len(rows) + len(METHODS) - 1
    fig, ax = plt.subplots(figsize=(11, max(5.6, 0.55 * plot_rows)))
    for y, (_, proportions) in zip(y_positions, rows):
        left = 0.0
        for category in (3, 2, 1):
            width = float(proportions[category])
            ax.barh(y, -width, left=left, color=colors[category], edgecolor='white', linewidth=0.5)
            left -= width
        left = 0.0
        for category in (4, 5, 6):
            width = float(proportions[category])
            ax.barh(y, width, left=left, color=colors[category], edgecolor='white', linewidth=0.5)
            left += width
    ax.axvline(0, color='0.25', linewidth=0.8)
    ax.set_yticks(y_positions)
    ax.set_yticklabels([label for label, _ in rows])
    ax.invert_yaxis()
    ax.set_xlabel('Percentage of responses (1–3 disagree; 4–6 agree)')
    ax.set_title(title)
    handles = [plt.Rectangle((0, 0), 1, 1, color=colors[value]) for value in categories]
    ax.legend(handles, [str(value) for value in categories], title='Response',
              ncol=6, loc='upper center', bbox_to_anchor=(0.5, -0.27))
    ax.grid(axis='x', alpha=0.2)
    fig.tight_layout(rect=(0, 0.27, 1, 1))
    plt.show()


## Sample demographics

**Method.** Age group, gender, dominant hand, study location, prior VR and teleport experience, gaming frequency, vision, and motion-sickness susceptibility are reported as textual frequency summaries. These variables describe the sample and are not subjected to hypothesis tests.

**Interpretation.** Demographic summaries document the composition and experience profile of the analyzed sample. They should not be interpreted as evidence of effects of the locomotion methods.


In [6]:
demography = raw['demography'].copy()
demography['ParticipantUID'] = participant_uid(
    demography['Location'], demography['Participant ID']
)
print(f"Participants: {demography['ParticipantUID'].nunique()}")

def age_lower_bound(label):
    digits = ''.join(
        character if character.isdigit() else ' ' for character in str(label)
    ).split()
    return int(digits[0]) if digits else 10_000

demographic_text_columns = [
    'Age', 'Gender', 'Dominant hand', 'Location',
    'How often have you used VR systems before this study?',
    'How familiar are you with teleport-based VR locomotion?',
    'How often do you play 3D video games (non-VR)?',
    'How is your vision?', 'Are you susceptible to motion sickness?',
]

MOTION_SICKNESS_SUSCEPTIBILITY_LABELS = {
    '1': ('1 — Very low:'),
    '2': ('2 — Low:'),
    '3': ('3 - Moderate'),
    '4': ('4 — High:'),
    '5': ('5 — Very high:'),
}

def demographic_response_label(column, response):
    if column != 'Are you susceptible to motion sickness?':
        return response
    normalized = re.sub(r'\.0$', '', str(response).strip())
    return MOTION_SICKNESS_SUSCEPTIBILITY_LABELS.get(normalized, response)

demographics_summary_rows = []
for column in demographic_text_columns:
    values = demography[column].fillna('Missing').astype(str)
    counts = values.value_counts(dropna=False)
    if column == 'Age':
        counts = counts.reindex(sorted(counts.index, key=age_lower_bound))
    elif column == 'Are you susceptible to motion sickness?':
        counts = counts.reindex(
            sorted(
                counts.index,
                key=lambda value: (
                    float(re.sub(r'\.0$', '', str(value)))
                    if re.fullmatch(r'[1-5](?:\.0)?', str(value).strip())
                    else math.inf
                ),
            ),
            fill_value=0,
        )
    print(f'\n{column}:')
    for response, count in counts.items():
        percent = 100 * count / len(demography)
        response_label = demographic_response_label(column, response)
        print(f'  {response_label}: {count} ({percent:.1f}%)')
        demographics_summary_rows.append({
            'Variable': column,
            'Category': response_label,
            'Count': int(count),
            'Percent': percent,
        })

demographics_summary = pd.DataFrame(demographics_summary_rows)


Participants: 42

Age:
  18–24: 8 (19.0%)
  25–34: 26 (61.9%)
  35–45: 7 (16.7%)
  46+: 1 (2.4%)

Gender:
  Male: 29 (69.0%)
  Female: 13 (31.0%)

Dominant hand:
  Right: 41 (97.6%)
  Left: 1 (2.4%)

Location:
  SITE1: 24 (57.1%)
  SITE2: 18 (42.9%)

How often have you used VR systems before this study?:
  1–5 times: 16 (38.1%)
  6–20 times: 9 (21.4%)
  More than 20 times: 9 (21.4%)
  Regular VR user (weekly or more): 4 (9.5%)
  Never: 4 (9.5%)

How familiar are you with teleport-based VR locomotion?:
  No prior experience: 18 (42.9%)
  Tried once or twice: 9 (21.4%)
  Used occasionally across multiple VR sessions: 9 (21.4%)
  Frequently used teleport locomotion: 6 (14.3%)

How often do you play 3D video games (non-VR)?:
  Rarely: 13 (31.0%)
  Weekly: 12 (28.6%)
  Never: 7 (16.7%)
  Daily: 6 (14.3%)
  Monthly: 4 (9.5%)

How is your vision?:
  I have normal vision (no glasses or contact lenses): 19 (45.2%)
  I wear glasses (right now): 19 (45.2%)
  I wear contact lenses (right now): 4 (

## RQ1 — Practical Viability

**How do different strategies for extending teleport locomotion support efficient long-range VR navigation?**

The hypothesis-directed analysis evaluates route-completion time. Precision-task performance, banner-inclusive time, and route-specific effects provide complementary exploratory or secondary evidence about practical viability.


### H1.1 — Navigation Task efficiency

**H1.1: Tsunami and Minimap-assisted teleportation will reduce route-completion times compared with Baseline teleportation during long-range VR navigation tasks.**

**Method.** Route times are first averaged within each participant and method. Each planned one-sided participant-level contrast uses a Wilcoxon signed-rank test with `zero_method='wilcox'`, no continuity correction, and automatic exact/asymptotic calculation. Holm correction controls the family-wise error rate across the two primary contrasts. Paired t-tests are reported only in the final sensitivity analysis.

**Interpretation.** A negative difference favours the extended method. H1.1 is supported for a contrast when the Holm-adjusted p-value is below `ALPHA` and the confidence interval is consistent with a reduction in time.


**Missing-data handling.** A participant × method summary is formed only when the analysed outcome has valid observations on Routes A, B, and C for that method. Subsequent pairing therefore excludes a participant only from contrasts involving an incomplete method. Reported `N` is the actual number of eligible pairs, not the overall study sample size.


In [7]:
h1_net = aggregate_method(city, 'NetPathTime')
rows = [
    paired_primary_result(h1_net, 'NetPathTime', 'tsunami', 'baseline', 'less', 'H1.1', 'primary net time'),
    paired_primary_result(h1_net, 'NetPathTime', 'map', 'baseline', 'less', 'H1.1', 'primary net time'),
]
show_results('H1.1_net', rows, adjust=True)


### Supplementary TotalPathTime

**Method.** The same one-sided participant-level Wilcoxon procedure is applied to total time, which includes banner delays. Holm correction covers the two comparisons with Baseline, and banner-delay summaries are reported descriptively. Paired t-tests appear only in the final sensitivity analysis.

**Interpretation.** Comparing net and total results indicates whether a navigation-time advantage remains after checkpoint interaction delays are included. A method may be fast in motion but lose that advantage through longer banner delays.


**Missing-data handling.** A participant × method summary is formed only when the analysed outcome has valid observations on Routes A, B, and C for that method. Subsequent pairing therefore excludes a participant only from contrasts involving an incomplete method. Reported `N` is the actual number of eligible pairs, not the overall study sample size.


In [8]:
h1_total = aggregate_method(city, 'TotalPathTime')
rows = [
    paired_primary_result(h1_total, 'TotalPathTime', 'tsunami', 'baseline', 'less', 'H1.1', 'secondary total time'),
    paired_primary_result(h1_total, 'TotalPathTime', 'map', 'baseline', 'less', 'H1.1', 'secondary total time'),
]
show_results('H1.1_total', rows, adjust=True)
display(city.groupby('Method')[['NetPathTime', 'BannerDelayTime', 'MeanBannerDelayTime', 'TotalPathTime']].mean().round(3))


          NetPathTime  BannerDelayTime  MeanBannerDelayTime  TotalPathTime
Method                                                                    
baseline       67.057           12.363                2.061         79.421
map            41.011           19.687                3.281         60.698
tsunami        48.658           13.293                2.216         61.950


## RQ1 — Supplementary and exploratory analyses


### Contextual descriptive outcome summary

**Method.** Means, standard deviations, and medians describe the observed navigation outcomes by method; no hypothesis test is performed in this cell.

**Interpretation.** Means summarize the centre, standard deviations describe variability, and medians provide a robust reference when distributions are skewed. Descriptive differences alone are not evidence of statistical significance; inferential comparisons follow below.


In [9]:
display(city.groupby('Method')[['NetPathTime', 'TotalPathTime', 'BannerDelayTime',
                                'TotalTeleports', 'TotalTraversedDistance']]
        .agg(['mean', 'std', 'median']).round(3))


         NetPathTime                  ... TotalTraversedDistance                  
                mean     std  median  ...                   mean      std   median
Method                                ...                                         
baseline      67.057  28.785  59.495  ...               3049.191  118.095  3047.30
map           41.011  16.141  37.245  ...               2964.645  338.948  2905.17
tsunami       48.658  17.628  44.790  ...               3198.617  393.988  3050.06

[3 rows x 15 columns]


### Contextual route-time visualization by method

Each dot represents one participant on one route. Boxes summarize the participant distribution for Baseline, Minimap, and Tsunami separately on routes A, B, and C. The left panel excludes banner delays; the right panel includes them.

**Method.** Boxplots show medians and interquartile ranges; jittered dots retain all participant-level observations.

**Interpretation.** Separation between boxes suggests a method difference, while overlap and outliers show uncertainty and heterogeneity. The plots are descriptive; inferential tests follow below.


In [10]:
route_time_summary = (
    city.groupby(['PathID', 'Method'])[['NetPathTime', 'TotalPathTime']]
        .agg(['count', 'mean', 'std', 'median'])
        .round(3)
)
display(route_time_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=False)
time_panels = (
    ('NetPathTime', 'Net time — banner delays excluded'),
    ('TotalPathTime', 'Total time — banner delays included'),
)

for ax, (outcome, title) in zip(axes, time_panels):
    sns.boxplot(
        data=city, x='PathID', y=outcome, hue='Method',
        order=ROUTES, hue_order=METHODS, palette=METHOD_COLORS,
        showfliers=False, width=0.72, linewidth=1.1, ax=ax,
    )
    sns.stripplot(
        data=city, x='PathID', y=outcome, hue='Method',
        order=ROUTES, hue_order=METHODS, palette=METHOD_COLORS,
        dodge=True, jitter=0.14, size=4, alpha=0.72,
        edgecolor='white', linewidth=0.35, legend=False, ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Route')
    ax.set_ylabel('Time (seconds)')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[:3], [METHOD_LABELS[m] for m in METHODS],
        title='Method', frameon=True,
    )

fig.suptitle('City Race completion times by route and locomotion method', y=1.02)
fig.tight_layout()


                NetPathTime                  ... TotalPathTime                
                      count    mean     std  ...          mean     std  median
PathID Method                                ...                              
A      baseline          42  74.133  29.609  ...        86.432  30.527  78.745
       map               42  37.466  12.171  ...        56.802  16.357  52.760
       tsunami           42  54.366  20.684  ...        67.982  23.778  62.030
B      baseline          42  58.503  17.576  ...        71.082  19.112  68.390
       map               42  42.352  17.326  ...        62.539  20.242  55.655
       tsunami           42  43.111  14.893  ...        56.056  17.415  52.110
C      baseline          42  68.536  34.814  ...        80.748  37.287  68.550
       map               42  43.216  18.063  ...        62.753  20.070  57.635
       tsunami           42  48.496  15.257  ...        61.814  17.840  59.110

[9 rows x 8 columns]


### RQ1 exploratory — Tsunami versus Minimap completion time

**Method.** For each time definition, a two-sided Wilcoxon signed-rank test compares the two long-range strategies after averaging the three routes within each participant and method. The Baseline contrasts are not repeated because they are already tested by H1.1.

**Interpretation.** A Holm-adjusted p-value below `ALPHA` indicates evidence of a difference between that pair of methods. The sign of the paired mean difference identifies which method was faster: a negative first-minus-second difference means that the first named method had the shorter time. These exploratory tests should be distinguished from the hypothesis-directed H1.1 comparisons.


**Missing-data handling.** A participant × method summary is formed only when the analysed outcome has valid observations on Routes A, B, and C for that method. Subsequent pairing therefore excludes a participant only from contrasts involving an incomplete method. Reported `N` is the actual number of eligible pairs, not the overall study sample size.


In [11]:
for outcome, label in [('NetPathTime', 'net time'), ('TotalPathTime', 'total time')]:
    frame = aggregate_method(city, outcome)
    pairwise_row = paired_primary_result(
        frame, outcome, 'tsunami', 'map', 'two-sided',
        'RQ1', f'exploratory long-range-strategy {label}',
    )
    show_results(f'RQ1_tsunami_vs_map_{outcome}', [pairwise_row])


### H1.1 supplementary — Route-level linear mixed-effects analysis

**Method.** All available participant × method × route observations are retained. The mixed-effects model permits an unbalanced number of repeated route observations, so a participant with a partially missing City Race route remains in this analysis with every valid observation. Separate linear mixed-effects models are fitted to log-transformed net and total completion time. Method, route (`PathID`), method order, and study location (Site 1/Site 2) are fixed effects; participant receives a random intercept. The derivative-free Powell optimizer is used because diagnostic fits showed that it avoids the singular-matrix failure produced by L-BFGS. Planned one-sided Tsunami–Baseline and Minimap–Baseline contrasts are Holm-adjusted within each time outcome. A likelihood-ratio comparison with an additional method × route interaction explores whether method effects vary across routes.

**Interpretation.** Exponentiated method coefficients are participant-adjusted time ratios relative to Baseline; values below 1 (negative percentages) favour the extended method. Agreement with the aggregated H1.1 analysis shows that its conclusion persists after retaining route-level variation and controlling for route, order, study location, and within-participant dependence. A significant interaction test suggests that the size of the method effect differs by route and should be interpreted together with the route-specific results below.


In [12]:
ANALYSIS_CONTEXT = {'Hypothesis': 'H1.1', 'Analysis': 'H1.1 supplementary — Route-level linear mixed-effects analysis'}
def route_level_mixed_analysis(frame, outcome):
    model_data = frame[[
        'ParticipantUID', 'Method', 'PathID', 'MethodOrder', 'Location', outcome
    ]].dropna().copy()
    if (model_data[outcome] <= 0).any():
        raise ValueError(f'{outcome} must be positive for log transformation.')
    model_data['LogTime'] = np.log(model_data[outcome])
    method_term = 'C(Method, Treatment(reference="baseline"))'
    additive_formula = (
        f'LogTime ~ {method_term} + C(PathID) + C(MethodOrder) + C(Location)'
    )
    no_method_formula = 'LogTime ~ C(PathID) + C(MethodOrder) + C(Location)'
    interaction_formula = (
        f'LogTime ~ {method_term} * C(PathID) + C(MethodOrder) + C(Location)'
    )
    no_method = smf.mixedlm(
        no_method_formula, model_data, groups=model_data['ParticipantUID'],
        re_formula='1',
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)
    additive = smf.mixedlm(
        additive_formula, model_data, groups=model_data['ParticipantUID'],
        re_formula='1',
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)
    interaction = smf.mixedlm(
        interaction_formula, model_data, groups=model_data['ParticipantUID'],
        re_formula='1',
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)

    contrast_rows = []
    for method in ('tsunami', 'map'):
        parameter = f'{method_term}[T.{method}]'
        beta = float(additive.params[parameter])
        se = float(additive.bse[parameter])
        z_value = beta / se
        ci_low, ci_high = beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * se
        contrast_rows.append({
            'Hypothesis': 'H1.1', 'Analysis': f'route-level {outcome}',
            'Outcome': outcome,
            'Contrast': f'{METHOD_LABELS[method]} - Baseline',
            'Alternative': 'less', 'NParticipants': model_data['ParticipantUID'].nunique(),
            'NObservations': len(model_data), 'NRouteObservations': len(model_data), 'LogCoefficient': beta,
            'AdjustedTimeRatio': np.exp(beta),
            'AdjustedPercentChange': 100 * (np.exp(beta) - 1),
            'RatioCI95Low': np.exp(ci_low), 'RatioCI95High': np.exp(ci_high),
            'ZValue': z_value, 'PValue': stats.norm.cdf(z_value),
            'TestMethod': 'Log-time LMM; participant random intercept',
            'Converged': bool(additive.converged),
            'RandomInterceptVariance': float(additive.cov_re.iloc[0, 0]),
        })

    lr_statistic = max(0.0, 2 * (interaction.llf - additive.llf))
    lr_df = int(interaction.df_modelwc - additive.df_modelwc)
    interaction_row = {
        'Outcome': outcome, 'Test': 'Method × route likelihood-ratio test',
        'LikelihoodRatio': lr_statistic, 'DegreesOfFreedom': lr_df,
        'PValue': stats.chi2.sf(lr_statistic, lr_df),
        'AdditiveConverged': bool(additive.converged),
        'InteractionConverged': bool(interaction.converged),
    }
    method_lr = max(0.0, 2 * (additive.llf - no_method.llf))
    method_df = int(additive.df_modelwc - no_method.df_modelwc)
    omnibus_row = {
        'Outcome': outcome, 'Test': 'Omnibus main effect of Method',
        'LikelihoodRatio': method_lr, 'DegreesOfFreedom': method_df,
        'PValue': stats.chi2.sf(method_lr, method_df),
        'ReducedConverged': bool(no_method.converged),
        'FullConverged': bool(additive.converged),
    }
    return contrast_rows, interaction_row, omnibus_row, np.asarray(additive.resid)

mixed_rows = []
interaction_rows = []
omnibus_rows = []
route_lmm_residuals = {}
for outcome in ('NetPathTime', 'TotalPathTime'):
    outcome_rows, interaction_row, omnibus_row, residuals = route_level_mixed_analysis(city, outcome)
    mixed_rows.extend(outcome_rows)
    interaction_rows.append(interaction_row)
    omnibus_rows.append(omnibus_row)
    route_lmm_residuals[outcome] = residuals

h1_route_mixed = pd.DataFrame(mixed_rows)
h1_route_mixed['PHolm'] = h1_route_mixed.groupby('Outcome')['PValue'].transform(
    lambda values: holm(values.to_numpy())
)
h1_route_interactions = pd.DataFrame(interaction_rows)
h1_route_omnibus = pd.DataFrame(omnibus_rows)
RESULTS['H1.1_route_level'] = h1_route_mixed
display(h1_route_mixed.style.format(precision=4))
display(h1_route_omnibus.style.format(precision=4))
display(h1_route_interactions.style.format(precision=4))
route_lmm_residual_diagnostics = residual_diagnostic_plot(
    route_lmm_residuals, 'H1.1 route-level log-time LMM residual Q–Q diagnostics'
)

triangulation_rows = []
for _, row in RESULTS['H1.1_net'].iterrows():
    triangulation_rows.append({
        'Approach': 'Participant-level mean of 3 routes',
        'Contrast': row['Contrast'], 'OutcomeScale': 'Seconds per route',
        'Estimate': row['MeanDifference'], 'CI95Low': row['CI95Low'],
        'CI95High': row['CI95High'], 'AdjustedP': row['PHolm'],
    })
for _, row in h1_route_mixed.loc[h1_route_mixed['Outcome'].eq('NetPathTime')].iterrows():
    triangulation_rows.append({
        'Approach': 'Route-level log-time LMM',
        'Contrast': row['Contrast'], 'OutcomeScale': 'Adjusted time ratio',
        'Estimate': row['AdjustedTimeRatio'], 'CI95Low': row['RatioCI95Low'],
        'CI95High': row['RatioCI95High'], 'AdjustedP': row['PHolm'],
    })
h1_triangulation = pd.DataFrame(triangulation_rows)
display(h1_triangulation.style.format(precision=4))


NetPathTime: Shapiro–Wilk p=0.0546, Q–Q r=0.996, skew=0.284; approximately normal residuals. Interpret the transformed LMM together with this diagnostic.
TotalPathTime: Shapiro–Wilk p=0.0759, Q–Q r=0.997, skew=0.296; approximately normal residuals. Interpret the transformed LMM together with this diagnostic.


Tsunami_quantitative_analysis.ipynb:code-cell-5:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### RQ1 exploratory — Route-specific pairwise time comparisons

**Method.** For each route and time definition, all three paired method contrasts are tested two-sided. Holm correction is applied across the nine comparisons within each time definition. These route-specific comparisons are post-hoc and should not replace the overall H1.1 test.

**Interpretation.** `MeanDifference` is the first named method minus the second: negative values favour the first method. `Significant=True` means the contrast remains below `ALPHA` after correction across all nine route-specific comparisons for that time definition.


In [13]:
route_pairwise_contrasts = (
    ('map', 'baseline'),
    ('tsunami', 'baseline'),
    ('tsunami', 'map'),
)

for outcome, result_key in (
    ('NetPathTime', 'Route_posthoc_net'),
    ('TotalPathTime', 'Route_posthoc_total'),
):
    route_rows = []
    for route in ROUTES:
        route_data = city.loc[
            city['PathID'].eq(route),
            ['ParticipantUID', 'Method', outcome],
        ]
        for first, second in route_pairwise_contrasts:
            result = paired_primary_result(
                route_data, outcome, first, second, 'two-sided',
                'Route post-hoc', f'Route {route}',
            )
            result['Route'] = route
            route_rows.append(result)

    route_table = pd.DataFrame(route_rows)
    route_table['PHolm'] = holm(route_table['PValue'].to_numpy())
    route_table['Significant'] = route_table['PHolm'] < ALPHA
    RESULTS[result_key] = route_table

    print(outcome)
    display(
        route_table[
            ['Route', 'Contrast', 'N', 'MeanFirst', 'MeanSecond',
             'MeanDifference', 'CI95Low', 'CI95High', 'PValue',
             'PHolm', 'Significant', 'TestMethod']
        ].style.format(precision=4)
    )


NetPathTime
TotalPathTime


### RQ1 exploratory — Interaction effort and path efficiency

**Method.** Total teleport counts are first screened with a participant-clustered Poisson GEE. When Pearson dispersion exceeds 1.5, the reported model is a negative-binomial regression that estimates its dispersion parameter from the data and uses participant-clustered robust standard errors. Path efficiency is defined as ideal route length divided by traversed distance. A log-scale random-intercept LMM is fitted first; if its random-intercept ICC is below 0.01, a cluster-robust fixed-effects linear model is used for inference and the LMM remains a sensitivity analysis. Two-sided method contrasts are Holm-adjusted within each outcome.

**Interpretation.** Exponentiated count contrasts are incidence-rate ratios; values below 1 indicate fewer teleport events. Exponentiated log-efficiency contrasts above 1 indicate more efficient paths. The dispersion estimate, random-effect ICC, and LMM-versus-cluster-robust sensitivity comparison are shown below. These analyses are exploratory rather than tests of H1.1.


In [14]:
ANALYSIS_CONTEXT = {'Hypothesis': 'RQ1', 'Analysis': 'RQ1 exploratory — Interaction effort and path efficiency'}
rq1_effort_rows = []
count_data = city.dropna(subset=[
    'TotalTeleports', 'ParticipantUID', 'Method', 'PathID', 'MethodOrder', 'Location'
]).copy()
count_formula = (
    'TotalTeleports ~ C(Method, Treatment(reference="baseline")) + '
    'C(PathID) + C(MethodOrder) + C(Location)'
)
poisson_gee = smf.gee(
    count_formula, groups='ParticipantUID', data=count_data,
    family=sm.families.Poisson(), cov_struct=sm.cov_struct.Exchangeable(),
).fit()
poisson_dispersion = float(
    np.sum(poisson_gee.resid_pearson ** 2) / poisson_gee.df_resid
)
if poisson_dispersion > 1.5:
    count_model = smf.negativebinomial(
        count_formula, data=count_data
    ).fit(
        disp=False, cov_type='cluster',
        cov_kwds={'groups': count_data['ParticipantUID']},
    )
    count_alpha = float(count_model.params['alpha'])
    count_model_name = (
        'Negative-binomial regression; estimated alpha; '
        'participant-clustered robust SE'
    )
else:
    count_model = poisson_gee
    count_alpha = 0.0
    count_model_name = 'Poisson GEE'

for method in ('map', 'tsunami'):
    parameter = f'C(Method, Treatment(reference="baseline"))[T.{method}]'
    beta, se = float(count_model.params[parameter]), float(count_model.bse[parameter])
    ci = beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * se
    rq1_effort_rows.append({
        'Outcome': 'TotalTeleports', 'Contrast': f'{METHOD_LABELS[method]} - Baseline',
        'EffectScale': 'Incidence-rate ratio', 'Estimate': np.exp(beta),
        'CI95Low': np.exp(ci[0]), 'CI95High': np.exp(ci[1]),
        'ZValue': beta / se, 'PValue': 2 * stats.norm.sf(abs(beta / se)),
        'TestMethod': count_model_name, 'PoissonPearsonDispersion': poisson_dispersion,
        'EstimatedNegativeBinomialAlpha': count_alpha,
    })

efficiency_data = city.dropna(subset=[
    'PathEfficiency', 'ParticipantUID', 'Method', 'PathID', 'MethodOrder', 'Location'
]).copy()
efficiency_data['LogPathEfficiency'] = np.log(efficiency_data['PathEfficiency'])
efficiency_formula = (
    'LogPathEfficiency ~ C(Method, Treatment(reference="baseline")) + '
    'C(PathID) + C(MethodOrder) + C(Location)'
)
outcome = 'PathEfficiency'
efficiency_lmm = smf.mixedlm(
    efficiency_formula, efficiency_data,
    groups=efficiency_data['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)
efficiency_random_variance = float(efficiency_lmm.cov_re.iloc[0, 0])
efficiency_residual_variance = float(efficiency_lmm.scale)
efficiency_icc = efficiency_random_variance / (
    efficiency_random_variance + efficiency_residual_variance
)
efficiency_cluster_ols = smf.ols(
    efficiency_formula, data=efficiency_data
).fit(
    cov_type='cluster', cov_kwds={'groups': efficiency_data['ParticipantUID']}
)
use_cluster_ols = efficiency_icc < 0.01
efficiency_primary_model = efficiency_cluster_ols if use_cluster_ols else efficiency_lmm
efficiency_primary_name = (
    'Log-efficiency OLS; participant-clustered robust SE '
    '(random-intercept ICC < .01)'
    if use_cluster_ols else 'Log-efficiency LMM'
)
efficiency_boundary_rows = []
for method in ('map', 'tsunami'):
    parameter = f'C(Method, Treatment(reference="baseline"))[T.{method}]'
    beta = float(efficiency_primary_model.params[parameter])
    se = float(efficiency_primary_model.bse[parameter])
    ci = beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * se
    rq1_effort_rows.append({
        'Outcome': 'PathEfficiency', 'Contrast': f'{METHOD_LABELS[method]} - Baseline',
        'EffectScale': 'Efficiency ratio', 'Estimate': np.exp(beta),
        'CI95Low': np.exp(ci[0]), 'CI95High': np.exp(ci[1]),
        'ZValue': beta / se, 'PValue': 2 * stats.norm.sf(abs(beta / se)),
        'TestMethod': efficiency_primary_name,
        'PoissonPearsonDispersion': np.nan,
        'EstimatedNegativeBinomialAlpha': np.nan,
    })
    lmm_beta, lmm_se = float(efficiency_lmm.params[parameter]), float(efficiency_lmm.bse[parameter])
    ols_beta, ols_se = float(efficiency_cluster_ols.params[parameter]), float(efficiency_cluster_ols.bse[parameter])
    efficiency_boundary_rows.append({
        'Outcome': 'PathEfficiency', 'Contrast': f'{METHOD_LABELS[method]} - Baseline',
        'RandomInterceptVariance': efficiency_random_variance,
        'ResidualVariance': efficiency_residual_variance, 'ICC': efficiency_icc,
        'PrimaryModel': efficiency_primary_name,
        'LMMEstimateRatio': np.exp(lmm_beta),
        'LMMPValue': 2 * stats.norm.sf(abs(lmm_beta / lmm_se)),
        'ClusterRobustOLSEstimateRatio': np.exp(ols_beta),
        'ClusterRobustOLSPValue': 2 * stats.norm.sf(abs(ols_beta / ols_se)),
        'SameDirection': np.sign(lmm_beta) == np.sign(ols_beta),
        'SameSignificanceAt05': (
            (2 * stats.norm.sf(abs(lmm_beta / lmm_se)) < ALPHA)
            == (2 * stats.norm.sf(abs(ols_beta / ols_se)) < ALPHA)
        ),
    })

rq1_effort_table = pd.DataFrame(rq1_effort_rows)
rq1_effort_table['PHolm'] = rq1_effort_table.groupby('Outcome')['PValue'].transform(
    lambda values: holm(values.to_numpy())
)
efficiency_boundary_sensitivity = pd.DataFrame(efficiency_boundary_rows)
RESULTS['RQ1_interaction_effort_path_efficiency'] = rq1_effort_table
display(rq1_effort_table.style.format(precision=4))
display(efficiency_boundary_sensitivity.style.format(precision=5))
display(city.groupby('Method')[['TotalTeleports', 'PathEfficiency']]
        .agg(['mean', 'std', 'median']).round(3))


         TotalTeleports                PathEfficiency              
                   mean     std median           mean    std median
Method                                                             
baseline         40.111  18.991   35.0          0.829  0.026  0.831
map               7.595   1.626    7.0          0.859  0.064  0.864
tsunami          14.690   3.312   14.0          0.798  0.073  0.819


### Precision Task (supplementary)

**Method.** Trial-level precision data are analysed with separate linear mixed-effects models for completion time, hold/aiming time, and 2D landing error. Method, target distance, their interaction, method order, and study location are fixed effects; participant receives a random intercept. Completion and hold time are log-transformed; landing error uses `log1p(error)`. The primary conditional-precision models exclude only trials flagged in `precision.csv` as raw-log-verified failed manipulations. A trial is not removed merely because it is an IQR outlier: it must also have an error of at least 25% of nominal distance and agreement between the processed trial and the original aim → teleport → appear telemetry. Manipulation-failure evidence is either (a) two in-range matching repetitions plus an error at least three times their median, (b) a verified aim-to-teleport interval below 0.25 s, conservatively interpreted as immediate/accidental confirmation, or (c) a repeated-failure group in which at least two raw-log-verified upper-IQR trials have errors of at least 25% of nominal distance. For a repeated-failure group, every raw-log-verified upper-IQR candidate in that participant × method × distance group is excluded. A short interval alone never excludes an otherwise accurate trial. The same LMMs are then fitted to all observations as a mandatory sensitivity analysis.

**Interpretation.** The cleaned models estimate precision conditional on a successfully understood and executed manipulation. The full-data models retain failed interactions and therefore better represent practical error tolerance and usability. A conclusion is robust only when its direction and inferential interpretation are consistent across both datasets. Because the exclusion rule was developed after inspecting interim data, both versions must be reported and the cleaned result must not be presented without the full-data sensitivity result.


**Timing note.** First-trial completion time starts at the first recorded `aim`, not at `appear_initial`; the preceding interval was the explanation phase and is intentionally excluded.

**Task geometry and ordering.** The processed task uses horizontal target centers at `(50, 0)`, `(100, 0)`, and `(200, 0)` m in the processed X--Z coordinate system (nominal azimuth 0 degrees along +X). Each method contains nine trials in three repeated 50--100--200 m cycles: trials 1/4/7, 2/5/8, and 3/6/9 correspond to 50, 100, and 200 m, respectively. Method order is the participant-level counterbalanced `MethodOrder` stored in `precision.csv`. The quantitative files do not independently encode the procedural reset/start-position implementation, so it is not inferred here.


In [15]:
ANALYSIS_CONTEXT = {'Hypothesis': 'RQ1', 'Analysis': 'RQ1 exploratory — Precision Mechanics'}
def precision_lmm(frame, outcome, transformation, analysis_label):
    model_data = frame[[
        'ParticipantUID', 'Method', 'Distance', 'MethodOrder', 'Location', outcome
    ]].dropna().copy()
    if transformation == 'log':
        if (model_data[outcome] <= 0).any():
            raise ValueError(f'{outcome} must be positive for log transformation.')
        model_data['ModelOutcome'] = np.log(model_data[outcome])
        transformed_scale = f'log({outcome})'
    elif transformation == 'log1p':
        if (model_data[outcome] < 0).any():
            raise ValueError(f'{outcome} must be non-negative for log1p transformation.')
        model_data['ModelOutcome'] = np.log1p(model_data[outcome])
        transformed_scale = f'log(1 + {outcome})'
    else:
        raise ValueError(transformation)

    method_term = 'C(Method, Treatment(reference="baseline"))'
    additive_formula = (
        f'ModelOutcome ~ {method_term} + C(Distance) + C(MethodOrder) + C(Location)'
    )
    no_method_formula = (
        'ModelOutcome ~ C(Distance) + C(MethodOrder) + C(Location)'
    )
    full_formula = (
        f'ModelOutcome ~ {method_term} * C(Distance) + C(MethodOrder) + C(Location)'
    )
    no_method = smf.mixedlm(
        no_method_formula, model_data, groups=model_data['ParticipantUID'],
        re_formula='1',
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)
    additive = smf.mixedlm(
        additive_formula, model_data, groups=model_data['ParticipantUID'],
        re_formula='1',
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)
    full = smf.mixedlm(
        full_formula, model_data, groups=model_data['ParticipantUID'],
        re_formula='1',
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)

    fixed_names = list(full.fe_params.index)
    fixed_covariance = full.cov_params().loc[fixed_names, fixed_names].to_numpy()
    design_info = full.model.data.design_info
    contrasts = []
    for distance in sorted(model_data['Distance'].unique()):
        # Empirical marginalization: standardize both methods over the same
        # observed MethodOrder and Location distribution. This removes the
        # previous dependence on whichever row happened to be first.
        standardization_rows = model_data[['MethodOrder', 'Location']].copy()
        standardization_rows['Distance'] = distance
        for first, second in (
            ('tsunami', 'baseline'), ('map', 'baseline'), ('tsunami', 'map')
        ):
            method_designs = {}
            method_predictions = {}
            method_gradients = {}
            for prediction_method in (first, second):
                new_rows = standardization_rows.copy()
                new_rows['Method'] = prediction_method
                design = np.asarray(build_design_matrices([design_info], new_rows)[0])
                linear = design @ full.fe_params.to_numpy()
                original = np.exp(linear)
                if transformation == 'log1p':
                    original = original - 1
                method_designs[prediction_method] = design
                method_predictions[prediction_method] = float(original.mean())
                method_gradients[prediction_method] = np.mean(
                    np.exp(linear)[:, None] * design, axis=0
                )
            vector = np.mean(method_designs[first] - method_designs[second], axis=0)
            estimate = float(vector @ full.fe_params.to_numpy())
            standard_error = float(np.sqrt(vector @ fixed_covariance @ vector))
            z_value = estimate / standard_error
            ci_low, ci_high = estimate + np.array([-1, 1]) * stats.norm.ppf(0.975) * standard_error
            original_difference = method_predictions[first] - method_predictions[second]
            original_gradient = method_gradients[first] - method_gradients[second]
            original_se = float(np.sqrt(
                original_gradient @ fixed_covariance @ original_gradient
            ))
            original_ci = original_difference + np.array([-1, 1]) * stats.norm.ppf(0.975) * original_se
            original_predictions = [method_predictions[first], method_predictions[second]]
            contrasts.append({
                'Hypothesis': 'Precision', 'Analysis': analysis_label,
                'Outcome': outcome, 'TransformedScale': transformed_scale,
                'Distance': int(distance),
                'Contrast': f'{METHOD_LABELS[first]} - {METHOD_LABELS[second]}',
                'NParticipants': model_data['ParticipantUID'].nunique(),
                'NTrials': len(model_data), 'LogScaleDifference': estimate,
                'Ratio': np.exp(estimate), 'RatioCI95Low': np.exp(ci_low),
                'RatioCI95High': np.exp(ci_high), 'ZValue': z_value,
                'AdjustedFirstOriginalScale': original_predictions[0],
                'AdjustedSecondOriginalScale': original_predictions[1],
                'AdjustedDifferenceOriginalScale': original_difference,
                'AdjustedDifferenceCI95Low': original_ci[0],
                'AdjustedDifferenceCI95High': original_ci[1],
                'PValue': 2 * stats.norm.sf(abs(z_value)),
                'TestMethod': 'Trial-level LMM pairwise contrast',
                'Converged': bool(full.converged),
                'RandomInterceptVariance': float(full.cov_re.iloc[0, 0]),
            })

    lr_statistic = max(0.0, 2 * (full.llf - additive.llf))
    lr_df = int(full.df_modelwc - additive.df_modelwc)
    interaction = {
        'Outcome': outcome, 'TransformedScale': transformed_scale,
        'Test': 'Method × distance likelihood-ratio test',
        'LikelihoodRatio': lr_statistic, 'DegreesOfFreedom': lr_df,
        'PValue': stats.chi2.sf(lr_statistic, lr_df),
        'AdditiveConverged': bool(additive.converged),
        'FullConverged': bool(full.converged),
        'MethodOmnibusLikelihoodRatio': max(0.0, 2 * (additive.llf - no_method.llf)),
        'MethodOmnibusDegreesOfFreedom': int(additive.df_modelwc - no_method.df_modelwc),
        'MethodOmnibusPValue': stats.chi2.sf(
            max(0.0, 2 * (additive.llf - no_method.llf)),
            int(additive.df_modelwc - no_method.df_modelwc),
        ),
        'NoMethodModelConverged': bool(no_method.converged),
    }
    residuals = np.asarray(full.resid)
    diagnostics = {
        'Outcome': outcome, 'TransformedScale': transformed_scale,
        'ResidualSkewness': stats.skew(residuals, bias=False),
        'ResidualExcessKurtosis': stats.kurtosis(residuals, fisher=True, bias=False),
        'ShapiroP': stats.shapiro(residuals).pvalue,
        'RandomInterceptVariance': float(full.cov_re.iloc[0, 0]),
        'ResidualVariance': float(full.scale),
    }
    diagnostics['ICC'] = diagnostics['RandomInterceptVariance'] / (
        diagnostics['RandomInterceptVariance'] + diagnostics['ResidualVariance']
    )
    return contrasts, interaction, diagnostics, residuals

precision_lmm_rows = []
precision_interactions = []
precision_lmm_diagnostics = []
precision_lmm_residuals = {}
for outcome, transformation in (
    ('CompletionTime', 'log'), ('HoldTime', 'log'), ('AimingError', 'log1p')
):
    rows, interaction, diagnostics, residuals = precision_lmm(
        precision_clean, outcome, transformation, 'cleaned trial-level LMM (primary)'
    )
    precision_lmm_rows.extend(rows)
    interaction['Dataset'] = 'cleaned (primary)'
    diagnostics['Dataset'] = 'cleaned (primary)'
    precision_interactions.append(interaction)
    precision_lmm_diagnostics.append(diagnostics)
    precision_lmm_residuals[f'{outcome} — cleaned'] = residuals

    full_rows, full_interaction, full_diagnostics, full_residuals = precision_lmm(
        precision_full, outcome, transformation, 'full-data trial-level LMM (sensitivity)'
    )
    precision_lmm_rows.extend(full_rows)
    full_interaction['Dataset'] = 'full data (sensitivity)'
    full_diagnostics['Dataset'] = 'full data (sensitivity)'
    precision_interactions.append(full_interaction)
    precision_lmm_diagnostics.append(full_diagnostics)
    precision_lmm_residuals[f'{outcome} — full'] = full_residuals

precision_lmm_table = pd.DataFrame(precision_lmm_rows)
precision_lmm_table['PHolm'] = precision_lmm_table.groupby(['Analysis', 'Outcome'])['PValue'].transform(
    lambda values: holm(values.to_numpy())
)
precision_interaction_table = pd.DataFrame(precision_interactions)
precision_lmm_diagnostic_table = pd.DataFrame(precision_lmm_diagnostics)
RESULTS['Precision_LMM'] = precision_lmm_table
display(precision_lmm_table.style.format(precision=4))
display(precision_interaction_table.style.format(precision=4))
display(precision_lmm_diagnostic_table.style.format(precision=4))
precision_residual_diagnostics = residual_diagnostic_plot(
    precision_lmm_residuals, 'Precision trial-level LMM residual Q–Q diagnostics'
)

print(f'Raw-log-verified failed-manipulation exclusions: {len(precision_full) - len(precision_clean)}')
display(precision.loc[
    precision['ExcludeFailedManipulation'],
    ['ParticipantUID', 'Method', 'Distance', 'Trial', 'AimingError',
     'HoldTime', 'RecommendedExclude', 'RepeatedFailureGroup',
     'RepeatedFailureTrialExclude', 'DecisionReason']
])
display(precision_clean.groupby(['Method', 'Distance'])[['CompletionTime', 'HoldTime', 'AimingError']]
        .agg(['mean', 'std']).round(3))
method_boxplot(precision_clean, 'AimingError',
               'Precision-task landing error — adjudicated dataset', 'Metres')


precision_200m = precision_clean.loc[precision_clean['Distance'].eq(200)].copy()
precision_200m['OffsetX'] = precision_200m['LandingPos_x'] + 68.3
precision_200m['OffsetZ'] = precision_200m['LandingPos_z']
precision_extent = max(
    6.0,
    float(np.nanquantile(np.abs(precision_200m[['OffsetX', 'OffsetZ']]), 0.99)) * 1.08,
)
fig, ax = plt.subplots(figsize=(5.8, 4.2))
for method in METHODS:
    subset = precision_200m.loc[precision_200m['Method'].eq(method)]
    ax.scatter(
        subset['OffsetX'], subset['OffsetZ'], s=16, alpha=0.48,
        color=METHOD_COLORS[method], label=METHOD_LABELS[method], edgecolors='none',
    )
ax.scatter([0], [0], marker='+', s=110, linewidth=2, color='black', label='Target')
for radius in (1, 2, 5):
    ax.add_patch(plt.Circle((0, 0), radius, fill=False, color='0.55',
                            linewidth=0.65, linestyle='--', alpha=0.65))
ax.axhline(0, color='0.75', linewidth=0.6)
ax.axvline(0, color='0.75', linewidth=0.6)
ax.set_xlim(-precision_extent, precision_extent)
ax.set_ylim(-precision_extent, precision_extent)
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Target-centred X offset (m)')
ax.set_ylabel('Target-centred Z offset (m)')
ax.set_title('200 m target')
ax.legend(frameon=True, fontsize=8, ncol=1, loc='center left',
          bbox_to_anchor=(1.03, 0.5))
fig.subplots_adjust(left=0.14, right=0.72, bottom=0.16, top=0.90)
plt.show()


CompletionTime — cleaned: Shapiro–Wilk p=0.0000, Q–Q r=0.890, skew=-3.852; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
CompletionTime — full: Shapiro–Wilk p=0.0000, Q–Q r=0.896, skew=-3.585; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
HoldTime — cleaned: Shapiro–Wilk p=0.0000, Q–Q r=0.910, skew=-2.575; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
HoldTime — full: Shapiro–Wilk p=0.0000, Q–Q r=0.913, skew=-2.409; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
AimingError — cleaned: Shapiro–Wilk p=0.0813, Q–Q r=0.999, skew=0.034; approximately normal residuals. Interpret the transformed LMM together with this diagnostic.
AimingError — full: Shapiro–Wilk p=0.0000, Q–Q r=0.977, skew=0.933; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
Raw-log-verified failed-manipulation

Tsunami_quantitative_analysis.ipynb:code-cell-5:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-15:217: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Navigation Task checkpoint precision (exploratory)

**Method.** This conditional ecological analysis uses successful direct-checkpoint sequences `teleport_map|teleport_tsunami → appear → progress_banner`. Landing position is measured at `progress_banner`, the first logged event that guarantees checkpoint activation; the earlier `appear` position is not used because Minimap can continue changing world position between these events without another teleport. Initial horizontal target distance is divided into two sufficiently populated bands, 200–350 and 350–500 m. A trial-level LMM predicts `log(1 + 2D landing error)` from method, distance band, their interaction, target identity, method order, and study location, with a participant random intercept. Two-sided Tsunami–Minimap contrasts are Holm-adjusted across the two bands. A complementary sensitivity model retains distance continuously (per 100 m) and tests the method × distance interaction.

**Interpretation.** A ratio below 1 indicates smaller landing error for Tsunami; a ratio above 1 indicates smaller error for Minimap. This dataset is conditioned on the first teleport directly activating the checkpoint and therefore does not estimate failed attempts or correction costs. The continuous-distance interaction indicates whether the between-method difference changes progressively with distance.


In [16]:
ANALYSIS_CONTEXT = {'Hypothesis': 'RQ1', 'Analysis': 'RQ1 exploratory — City Race direct-checkpoint precision (200–500 m)'}
city_precision_summary = (
    city_precision.groupby(['Method', 'DistanceBandAnalysis'], observed=True)['LandingError2D']
    .agg(['count', 'mean', 'std', 'median', 'max']).round(3)
)
display(city_precision_summary)

fig, axes = plt.subplots(
    1, len(CITY_PRECISION_BANDS),
    figsize=(6 * len(CITY_PRECISION_BANDS), 5), sharey=True,
)
axes = np.atleast_1d(axes)
for ax, band in zip(axes, CITY_PRECISION_BANDS):
    subset = city_precision.loc[city_precision['DistanceBandAnalysis'].eq(band)]
    sns.boxplot(
        data=subset, x='Method', y='LandingError2D', order=['map', 'tsunami'],
        hue='Method', palette=METHOD_COLORS, dodge=False, showfliers=False, ax=ax,
    )
    if ax.get_legend() is not None: ax.get_legend().remove()
    sns.stripplot(
        data=subset, x='Method', y='LandingError2D', order=['map', 'tsunami'],
        color='black', alpha=0.3, jitter=0.2, size=3, ax=ax,
    )
    ax.set_title(f'{band} m')
    ax.set_xlabel('Method')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Minimap', 'Tsunami'])
    ax.set_ylabel('2D landing error (m)' if ax is axes[0] else '')
fig.suptitle('City Race direct-checkpoint landing error — two distance bands', y=1.02)
fig.tight_layout(); plt.show()

outcome = 'LandingError2D'
city_precision_model = city_precision.dropna(subset=[
    'ParticipantUID', 'Method', 'DistanceBandAnalysis', 'TargetID',
    'MethodOrder', 'Location', 'Log1pLandingError2D'
]).copy()
method_term = 'C(Method, Treatment(reference="map"))'
band_term = 'C(DistanceBandAnalysis, Treatment(reference="200-350"))'
additive_formula = (
    f'Log1pLandingError2D ~ {method_term} + {band_term} + '
    'C(TargetID) + C(MethodOrder) + C(Location)'
)
no_method_formula = (
    f'Log1pLandingError2D ~ {band_term} + '
    'C(TargetID) + C(MethodOrder) + C(Location)'
)
interaction_formula = (
    f'Log1pLandingError2D ~ {method_term} * {band_term} + '
    'C(TargetID) + C(MethodOrder) + C(Location)'
)
city_precision_no_method = smf.mixedlm(
    no_method_formula, city_precision_model,
    groups=city_precision_model['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)
city_precision_additive = smf.mixedlm(
    additive_formula, city_precision_model,
    groups=city_precision_model['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)
city_precision_interaction = smf.mixedlm(
    interaction_formula, city_precision_model,
    groups=city_precision_model['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)

fixed_names = list(city_precision_interaction.fe_params.index)
fixed_covariance = city_precision_interaction.cov_params().loc[
    fixed_names, fixed_names
].to_numpy()
design_info = city_precision_interaction.model.data.design_info
city_precision_contrasts = []
for band in CITY_PRECISION_BANDS:
    # Standardize over the pooled observed target/order/site distribution
    # within this distance band instead of predicting at the first row.
    standardization_rows = city_precision_model.loc[
        city_precision_model['DistanceBandAnalysis'].eq(band),
        ['TargetID', 'MethodOrder', 'Location'],
    ].copy()
    designs = {}
    predictions = {}
    gradients = {}
    for prediction_method in ('tsunami', 'map'):
        new_rows = standardization_rows.copy()
        new_rows['Method'] = prediction_method
        new_rows['DistanceBandAnalysis'] = band
        design = np.asarray(build_design_matrices([design_info], new_rows)[0])
        linear = design @ city_precision_interaction.fe_params.to_numpy()
        original = np.exp(linear) - 1
        designs[prediction_method] = design
        predictions[prediction_method] = float(original.mean())
        gradients[prediction_method] = np.mean(
            np.exp(linear)[:, None] * design, axis=0
        )
    vector = np.mean(designs['tsunami'] - designs['map'], axis=0)
    estimate = float(vector @ city_precision_interaction.fe_params.to_numpy())
    standard_error = float(np.sqrt(vector @ fixed_covariance @ vector))
    z_value = estimate / standard_error
    ci_low, ci_high = estimate + np.array([-1, 1]) * stats.norm.ppf(0.975) * standard_error
    original_predictions = [predictions['tsunami'], predictions['map']]
    original_difference = predictions['tsunami'] - predictions['map']
    original_gradient = gradients['tsunami'] - gradients['map']
    original_se = float(np.sqrt(
        original_gradient @ fixed_covariance @ original_gradient
    ))
    original_ci = original_difference + np.array([-1, 1]) * stats.norm.ppf(0.975) * original_se
    city_precision_contrasts.append({
        'Hypothesis': 'City Race precision', 'Analysis': 'exploratory ecological LMM',
        'Outcome': 'LandingError2D', 'DistanceBand': band,
        'Contrast': 'Tsunami - Minimap',
        'NParticipants': city_precision_model['ParticipantUID'].nunique(),
        'NEvents': len(city_precision_model), 'Log1pDifference': estimate,
        'ErrorPlusOneRatio': np.exp(estimate),
        'RatioCI95Low': np.exp(ci_low), 'RatioCI95High': np.exp(ci_high),
        'AdjustedTsunamiErrorM': original_predictions[0],
        'AdjustedMinimapErrorM': original_predictions[1],
        'AdjustedDifferenceM': original_difference,
        'AdjustedDifferenceCI95LowM': original_ci[0],
        'AdjustedDifferenceCI95HighM': original_ci[1],
        'ZValue': z_value, 'PValue': 2 * stats.norm.sf(abs(z_value)),
        'TestMethod': 'Trial-level log1p-error LMM',
    })
city_precision_lmm_table = pd.DataFrame(city_precision_contrasts)
city_precision_lmm_table['PHolm'] = holm(city_precision_lmm_table['PValue'])
RESULTS['City_Race_precision_LMM'] = city_precision_lmm_table
display(city_precision_lmm_table.style.format(precision=4))

lr_statistic = max(0.0, 2 * (city_precision_interaction.llf - city_precision_additive.llf))
lr_df = int(city_precision_interaction.df_modelwc - city_precision_additive.df_modelwc)
city_precision_interaction_test = pd.DataFrame([{
    'Test': 'Method × distance-band likelihood-ratio test',
    'LikelihoodRatio': lr_statistic, 'DegreesOfFreedom': lr_df,
    'PValue': stats.chi2.sf(lr_statistic, lr_df),
    'AdditiveConverged': bool(city_precision_additive.converged),
    'InteractionConverged': bool(city_precision_interaction.converged),
    'MethodOmnibusLikelihoodRatio': max(
        0.0, 2 * (city_precision_additive.llf - city_precision_no_method.llf)
    ),
    'MethodOmnibusDegreesOfFreedom': int(
        city_precision_additive.df_modelwc - city_precision_no_method.df_modelwc
    ),
    'MethodOmnibusPValue': stats.chi2.sf(
        max(0.0, 2 * (city_precision_additive.llf - city_precision_no_method.llf)),
        int(city_precision_additive.df_modelwc - city_precision_no_method.df_modelwc),
    ),
}])
display(city_precision_interaction_test.style.format(precision=4))
city_precision_residual_diagnostics = residual_diagnostic_plot(
    {'City Race LandingError2D': np.asarray(city_precision_interaction.resid)},
    'City Race precision LMM residual Q–Q diagnostic',
)

def city_precision_scatter(frame):
    common_limit = 1.08 * max(
        frame['LandingOffset_x'].abs().max(), frame['LandingOffset_z'].abs().max()
    )
    band_count = len(CITY_PRECISION_BANDS)
    fig, axes = plt.subplots(
        band_count, 2, figsize=(10, 4.3 * band_count),
        sharex=True, sharey=True,
    )
    axes = np.atleast_2d(axes)
    for row, band in enumerate(CITY_PRECISION_BANDS):
        for column, method in enumerate(('map', 'tsunami')):
            ax = axes[row, column]
            subset = frame.loc[
                frame['DistanceBandAnalysis'].eq(band) & frame['Method'].eq(method)
            ]
            ax.scatter(subset['LandingOffset_x'], subset['LandingOffset_z'],
                       alpha=0.45, s=22, color=METHOD_COLORS[method])
            ax.scatter([0], [0], marker='+', s=120, linewidths=2, color='crimson')
            ax.axhline(0, color='0.7', linewidth=0.7); ax.axvline(0, color='0.7', linewidth=0.7)
            ax.set_xlim(-common_limit, common_limit); ax.set_ylim(-common_limit, common_limit)
            ax.set_aspect('equal', adjustable='box')
            if row == 0: ax.set_title(METHOD_LABELS[method])
            if column == 0: ax.set_ylabel(f'{band} m\nZ offset (m)')
            if row == band_count - 1: ax.set_xlabel('X offset (m)')
    fig.suptitle('City Race target-centred landing positions', y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.98]); plt.show()

city_precision_scatter(city_precision)


# Continuous-distance sensitivity: distance is centred and expressed per 100 m.
city_precision_continuous_formula = (
    'Log1pLandingError2D ~ C(Method, Treatment(reference="map")) * '
    'Distance100Centered + C(TargetID) + C(MethodOrder) + C(Location)'
)
city_precision_continuous = smf.mixedlm(
    city_precision_continuous_formula, city_precision_model,
    groups=city_precision_model['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)
continuous_interaction_parameter = (
    'C(Method, Treatment(reference="map"))[T.tsunami]:Distance100Centered'
)
continuous_beta = float(city_precision_continuous.params[continuous_interaction_parameter])
continuous_se = float(city_precision_continuous.bse[continuous_interaction_parameter])
continuous_ci = continuous_beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * continuous_se
city_precision_continuous_table = pd.DataFrame([{
    'Hypothesis': 'City Race precision',
    'Analysis': 'direct-hit continuous-distance sensitivity',
    'Outcome': 'LandingError2D',
    'Contrast': 'Method × distance (Tsunami vs Minimap, per 100 m)',
    'NParticipants': city_precision_model['ParticipantUID'].nunique(),
    'NEvents': len(city_precision_model),
    'Log1pInteraction': continuous_beta,
    'MultiplicativeInteraction': np.exp(continuous_beta),
    'InteractionCI95Low': np.exp(continuous_ci[0]),
    'InteractionCI95High': np.exp(continuous_ci[1]),
    'ZValue': continuous_beta / continuous_se,
    'PValue': 2 * stats.norm.sf(abs(continuous_beta / continuous_se)),
    'TestMethod': 'Trial-level log1p-error LMM with continuous distance',
    'Converged': bool(city_precision_continuous.converged),
}])
RESULTS['City_Race_precision_continuous_LMM'] = city_precision_continuous_table
display(city_precision_continuous_table.style.format(precision=4))


                              count   mean    std  median    max
Method  DistanceBandAnalysis                                    
map     200-350                 124  2.710  1.302   2.535  5.991
        350-500                 576  2.819  1.347   2.720  6.788
tsunami 200-350                  52  4.216  2.125   3.997  9.502
        350-500                 118  4.543  1.981   4.321  9.476
City Race LandingError2D: Shapiro–Wilk p=0.0000, Q–Q r=0.988, skew=-0.573; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.


Tsunami_quantitative_analysis.ipynb:code-cell-16:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-16:175: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### RQ1 exploratory — Checkpoint-approach precision and correction episodes

**Method.** This complementary analysis treats the complete approach to a checkpoint as one episode. It includes a 200–500 m initial target distance followed either by a direct hit or by one or more correction teleports. The primary geometric outcome is 2D error after the first long teleport, before any correction. Two distance bands (200–350 and 350–500 m) are used for method contrasts in a `log(1 + error)` LMM with method, band, their interaction, target identity, method order, study location, and participant random intercept. A continuous-distance sensitivity model tests method × distance per 100 m. Correction counts and distances are descriptive interaction-cost outcomes because sequential approximation is part of Tsunami's interaction design.

**Interpretation.** Unlike the direct-hit dataset, this analysis retains approaches that required correction and is therefore the less selection-sensitive ecological precision analysis. Lower first-landing error indicates that the initial long teleport ended closer to the checkpoint. Correction counts should be interpreted as interaction cost, not as a pure accuracy failure rate.


In [17]:
ANALYSIS_CONTEXT = {'Hypothesis': 'RQ1', 'Analysis': 'RQ1 exploratory — Checkpoint-approach precision and correction episodes'}
episode_summary = (
    checkpoint_episodes.groupby(
        ['Method', 'DistanceBandAnalysis'], observed=True
    )['FirstLandingError2D']
    .agg(['count', 'mean', 'std', 'median', 'max']).round(3)
)
display(episode_summary)

episode_correction_summary = checkpoint_episodes.groupby('Method').agg(
    Episodes=('ParticipantUID', 'size'),
    Participants=('ParticipantUID', 'nunique'),
    DirectHits=('DirectHit', 'sum'),
    MeanCorrections=('CorrectionCount', 'mean'),
    MedianCorrections=('CorrectionCount', 'median'),
    MeanTotalCorrectionDistance2D=('TotalCorrectionDistance2D', 'mean'),
).round(3)
display(episode_correction_summary)

fig, axes = plt.subplots(
    1, len(CITY_PRECISION_BANDS),
    figsize=(6 * len(CITY_PRECISION_BANDS), 5), sharey=True,
)
axes = np.atleast_1d(axes)
for ax, band in zip(axes, CITY_PRECISION_BANDS):
    subset = checkpoint_episodes.loc[
        checkpoint_episodes['DistanceBandAnalysis'].eq(band)
    ]
    sns.boxplot(
        data=subset, x='Method', y='FirstLandingError2D',
        order=['map', 'tsunami'], hue='Method', palette=METHOD_COLORS,
        dodge=False, showfliers=False, ax=ax,
    )
    if ax.get_legend() is not None: ax.get_legend().remove()
    sns.stripplot(
        data=subset, x='Method', y='FirstLandingError2D',
        order=['map', 'tsunami'], color='black', alpha=0.22,
        jitter=0.2, size=2.5, ax=ax,
    )
    ax.set_title(f'{band} m')
    ax.set_xlabel('Method')
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Minimap', 'Tsunami'])
    ax.set_ylabel('First-landing 2D error (m)' if ax is axes[0] else '')
fig.suptitle('Checkpoint approaches: error after the first long teleport', y=1.02)
fig.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(8.5, 3.8))
for ax, outcome, title, ylabel in (
    (axes[0], 'FirstLandingError2D', 'First landing error — log scale', 'Error (m)'),
    (axes[1], 'CorrectionCount', 'Corrective teleport count — symlog scale', 'Corrections'),
):
    sns.boxplot(
        data=checkpoint_episodes, x='DistanceBandAnalysis', y=outcome, hue='Method',
        order=CITY_PRECISION_BANDS, hue_order=['map', 'tsunami'],
        palette=METHOD_COLORS, showfliers=False, ax=ax,
    )
    sns.stripplot(
        data=checkpoint_episodes, x='DistanceBandAnalysis', y=outcome, hue='Method',
        order=CITY_PRECISION_BANDS, hue_order=['map', 'tsunami'],
        palette=METHOD_COLORS, dodge=True, jitter=0.15, alpha=0.35,
        size=2.5, legend=False, ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel('Initial distance')
    ax.set_ylabel(ylabel)
axes[0].set_yscale('log')
axes[1].set_yscale('symlog', linthresh=1)
axes[0].get_legend().remove()
handles, _ = axes[1].get_legend_handles_labels()
axes[1].get_legend().remove()
fig.legend(
    handles[:2], ['Minimap', 'Tsunami'], title='Method', ncol=2,
    loc='lower center', bbox_to_anchor=(0.5, -0.01),
)
fig.tight_layout(rect=(0, 0.14, 1, 1))
plt.show()

episode_model_data = checkpoint_episodes.dropna(subset=[
    'ParticipantUID', 'Method', 'DistanceBandAnalysis', 'TargetID',
    'MethodOrder', 'Location', 'Log1pFirstLandingError2D',
    'Distance100Centered',
]).copy()
episode_method_term = 'C(Method, Treatment(reference="map"))'
episode_band_term = 'C(DistanceBandAnalysis, Treatment(reference="200-350"))'
episode_formula = (
    f'Log1pFirstLandingError2D ~ {episode_method_term} * {episode_band_term} + '
    'C(TargetID) + C(MethodOrder) + C(Location)'
)
episode_band_model = smf.mixedlm(
    episode_formula, episode_model_data,
    groups=episode_model_data['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)

episode_fixed_names = list(episode_band_model.fe_params.index)
episode_cov = episode_band_model.cov_params().loc[
    episode_fixed_names, episode_fixed_names
].to_numpy()
episode_design_info = episode_band_model.model.data.design_info
episode_contrasts = []
for band in CITY_PRECISION_BANDS:
    standardization_rows = episode_model_data.loc[
        episode_model_data['DistanceBandAnalysis'].eq(band),
        ['TargetID', 'MethodOrder', 'Location'],
    ].copy()
    method_designs = {}
    method_predictions = {}
    for prediction_method in ('tsunami', 'map'):
        new_rows = standardization_rows.copy()
        new_rows['Method'] = prediction_method
        new_rows['DistanceBandAnalysis'] = band
        design = np.asarray(build_design_matrices([episode_design_info], new_rows)[0])
        linear = design @ episode_band_model.fe_params.to_numpy()
        method_designs[prediction_method] = design
        method_predictions[prediction_method] = float((np.exp(linear) - 1).mean())
    vector = np.mean(method_designs['tsunami'] - method_designs['map'], axis=0)
    estimate = float(vector @ episode_band_model.fe_params.to_numpy())
    standard_error = float(np.sqrt(vector @ episode_cov @ vector))
    ci = estimate + np.array([-1, 1]) * stats.norm.ppf(0.975) * standard_error
    predictions = [method_predictions['tsunami'], method_predictions['map']]
    episode_contrasts.append({
        'Hypothesis': 'City Race checkpoint approach precision',
        'Analysis': 'complete-episode ecological LMM',
        'Outcome': 'FirstLandingError2D', 'DistanceBand': band,
        'Contrast': 'Tsunami - Minimap',
        'NParticipants': episode_model_data['ParticipantUID'].nunique(),
        'NEvents': len(episode_model_data),
        'Log1pDifference': estimate,
        'ErrorPlusOneRatio': np.exp(estimate),
        'RatioCI95Low': np.exp(ci[0]), 'RatioCI95High': np.exp(ci[1]),
        'AdjustedTsunamiErrorM': predictions[0],
        'AdjustedMinimapErrorM': predictions[1],
        'AdjustedDifferenceM': predictions[0] - predictions[1],
        'ZValue': estimate / standard_error,
        'PValue': 2 * stats.norm.sf(abs(estimate / standard_error)),
        'TestMethod': 'Episode-level log1p-error LMM',
        'Converged': bool(episode_band_model.converged),
    })
checkpoint_episode_lmm_table = pd.DataFrame(episode_contrasts)
checkpoint_episode_lmm_table['PHolm'] = holm(
    checkpoint_episode_lmm_table['PValue'].to_numpy()
)
RESULTS['City_Race_checkpoint_episodes_LMM'] = checkpoint_episode_lmm_table
display(checkpoint_episode_lmm_table.style.format(precision=4))

episode_continuous_formula = (
    'Log1pFirstLandingError2D ~ C(Method, Treatment(reference="map")) * '
    'Distance100Centered + C(TargetID) + C(MethodOrder) + C(Location)'
)
episode_continuous_model = smf.mixedlm(
    episode_continuous_formula, episode_model_data,
    groups=episode_model_data['ParticipantUID'], re_formula='1',
).fit(reml=False, method='powell', maxiter=5000, disp=False)
episode_interaction_parameter = (
    'C(Method, Treatment(reference="map"))[T.tsunami]:Distance100Centered'
)
episode_beta = float(episode_continuous_model.params[episode_interaction_parameter])
episode_se = float(episode_continuous_model.bse[episode_interaction_parameter])
episode_ci = episode_beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * episode_se
checkpoint_episode_continuous_table = pd.DataFrame([{
    'Hypothesis': 'City Race checkpoint approach precision',
    'Analysis': 'complete-episode continuous-distance sensitivity',
    'Outcome': 'FirstLandingError2D',
    'Contrast': 'Method × distance (Tsunami vs Minimap, per 100 m)',
    'NParticipants': episode_model_data['ParticipantUID'].nunique(),
    'NEvents': len(episode_model_data),
    'Log1pInteraction': episode_beta,
    'MultiplicativeInteraction': np.exp(episode_beta),
    'InteractionCI95Low': np.exp(episode_ci[0]),
    'InteractionCI95High': np.exp(episode_ci[1]),
    'ZValue': episode_beta / episode_se,
    'PValue': 2 * stats.norm.sf(abs(episode_beta / episode_se)),
    'TestMethod': 'Episode-level log1p-error LMM with continuous distance',
    'Converged': bool(episode_continuous_model.converged),
}])
RESULTS['City_Race_checkpoint_episodes_continuous_LMM'] = (
    checkpoint_episode_continuous_table
)
display(checkpoint_episode_continuous_table.style.format(precision=4))


                              count    mean     std  median      max
Method  DistanceBandAnalysis                                        
map     200-350                 124   5.453  27.998   2.729  314.165
        350-500                 617   7.030  37.339   2.915  613.327
tsunami 200-350                 125  20.160  38.897  11.017  281.061
        350-500                 626  42.959  72.471  18.854  455.064
         Episodes  ...  MeanTotalCorrectionDistance2D
Method             ...                               
map           741  ...                          6.611
tsunami       751  ...                         45.049

[2 rows x 6 columns]


Tsunami_quantitative_analysis.ipynb:code-cell-17:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-17:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## RQ2 — Re-engagement and Perceived Spatial Awareness

**How do the evaluated world-centered Tsunami and controller-mounted Minimap implementations differ in post-teleport re-engagement and perceived spatial awareness?**


### H2.1 — Post-teleport reorientation effort

**H2.1: Tsunami will reduce post-teleport reorientation effort compared with Minimap-assisted teleportation.**

**Operationalization.** The primary temporal behavioral outcome is landing-to-confirmation latency (`BannerDelayTime`), measured from `appear` to `progress_banner`. The supporting angular behavioral outcome is head rotation during landing-to-confirmation (`HeadRotation3D`) over the same interval. Both outcomes are modeled using $\log(1+x)$ mixed-effects models with Minimap as the method reference, participant random intercepts, and fixed effects for environmental target, method order, and study site. Holm correction covers these two outcomes.

Raw arithmetic means are reported descriptively. Inferential effects are model-adjusted marginal contrasts converted to the original measurement scale. Consequently, the adjusted contrasts need not equal the arithmetic differences between the raw condition means.

These measures capture behavioral re-engagement, visual search, and interaction-transition behavior. They do not directly measure spatial cognition, spatial updating, acquired spatial knowledge, or route learning.


In [18]:
ANALYSIS_CONTEXT = {'Hypothesis': 'H2.1', 'Analysis': 'H2.1 — Post-teleport reorientation effort'}
H21_PUBLICATION_LABELS = {
    'BannerDelayTime': 'landing-to-confirmation latency',
    'HeadRotation3D': 'head rotation during landing-to-confirmation',
}

def h21_event_lmm(frame, outcome, role, alternative='less'):
    model_data = frame.loc[
        frame['Method'].isin(['map', 'tsunami'])
    ].dropna(subset=[
        outcome, 'ParticipantUID', 'Method', 'TargetID', 'MethodOrder', 'Location'
    ]).copy()
    model_data['ModelOutcome'] = np.log1p(model_data[outcome])
    formula = (
        'ModelOutcome ~ C(Method, Treatment(reference="map")) + '
        'C(TargetID) + C(MethodOrder) + C(Location)'
    )
    model = smf.mixedlm(
        formula, model_data, groups=model_data['ParticipantUID'], re_formula='1'
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)
    parameter = 'C(Method, Treatment(reference="map"))[T.tsunami]'
    beta = float(model.params[parameter])
    se = float(model.bse[parameter])
    z_value = beta / se
    if alternative == 'less':
        p_value = stats.norm.cdf(z_value)
    elif alternative == 'greater':
        p_value = stats.norm.sf(z_value)
    elif alternative == 'two-sided':
        p_value = 2 * stats.norm.sf(abs(z_value))
    else:
        raise ValueError(alternative)
    ci = beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * se
    # Empirical marginalization (standardization): predict every observed
    # event once under each method while retaining its observed target, order,
    # and site covariates; then average fixed-effect predictions after
    # back-transforming with exp(eta)-1.
    prediction_design = {}
    marginal_predictions = {}
    marginal_gradients = {}
    fixed_names = list(model.fe_params.index)
    fixed_covariance = model.cov_params().loc[fixed_names, fixed_names].to_numpy()
    for prediction_method in ('tsunami', 'map'):
        prediction_rows = model_data[
            ['TargetID', 'MethodOrder', 'Location']
        ].copy()
        prediction_rows['Method'] = prediction_method
        design = np.asarray(build_design_matrices(
            [model.model.data.design_info], prediction_rows
        )[0])
        linear = design @ model.fe_params.to_numpy()
        original = np.exp(linear) - 1
        prediction_design[prediction_method] = design
        marginal_predictions[prediction_method] = float(original.mean())
        marginal_gradients[prediction_method] = np.mean(
            np.exp(linear)[:, None] * design, axis=0
        )
    original_difference = (
        marginal_predictions['tsunami'] - marginal_predictions['map']
    )
    original_gradient = (
        marginal_gradients['tsunami'] - marginal_gradients['map']
    )
    original_se = float(np.sqrt(
        original_gradient @ fixed_covariance @ original_gradient
    ))
    original_ci = original_difference + np.array([-1, 1]) * stats.norm.ppf(0.975) * original_se
    result = {
        'Hypothesis': 'H2.1', 'OutcomeRole': role, 'Outcome': outcome,
        'PublicationOutcome': H21_PUBLICATION_LABELS.get(outcome, outcome),
        'Contrast': 'Tsunami - Minimap', 'Alternative': alternative,
        'NParticipants': model_data['ParticipantUID'].nunique(),
        'NEvents': len(model_data), 'Log1pDifference': beta,
        'AdjustedOutcomePlusOneRatio': np.exp(beta),
        'RatioCI95Low': np.exp(ci[0]), 'RatioCI95High': np.exp(ci[1]),
        'AdjustedTsunamiOriginalScale': marginal_predictions['tsunami'],
        'AdjustedMinimapOriginalScale': marginal_predictions['map'],
        'AdjustedDifferenceOriginalScale': original_difference,
        'AdjustedDifferenceCI95Low': original_ci[0],
        'AdjustedDifferenceCI95High': original_ci[1],
        'ZValue': z_value, 'PValue': p_value,
        'TestMethod': 'Event-level log1p LMM; participant random intercept',
        'Converged': bool(model.converged),
        'RandomInterceptVariance': float(model.cov_re.iloc[0, 0]),
    }
    return result, np.asarray(model.resid)

h21_rows = []
h21_residuals = {}
for outcome, role in (
    ('BannerDelayTime', 'primary temporal behavioral outcome'),
    ('HeadRotation3D', 'supporting angular behavioral outcome'),
):
    result, residuals = h21_event_lmm(orientation_events, outcome, role)
    result['MeasurementInterval'] = 'appear → progress_banner'
    h21_rows.append(result)
    h21_residuals[outcome] = residuals
h21_lmm_table = pd.DataFrame(h21_rows)
h21_lmm_table['PHolm'] = holm(h21_lmm_table['PValue'])
RESULTS['H2.1_event_level_LMM'] = h21_lmm_table
display(h21_lmm_table.style.format(precision=4))
h21_residual_diagnostics = residual_diagnostic_plot(
    h21_residuals, 'H2.1 — transformed LMM residual diagnostics',
)

h21_participant_rows = []
for outcome, role in (
    ('BannerDelayTime', 'primary temporal behavioral outcome'),
    ('HeadRotation3D', 'supporting angular behavioral outcome'),
):
    participant_means = aggregate_method(orientation_events, outcome)
    h21_participant_rows.append(paired_primary_result(
        participant_means, outcome, 'tsunami', 'map', 'less',
        'H2.1', f'{role} participant-mean sensitivity',
    ))
show_results('H2.1_participant_mean_sensitivity', h21_participant_rows, adjust=True)

display(orientation_events.groupby('Method')[
    ['BannerDelayTime', 'HeadRotation3D']
].agg(['count', 'mean', 'std', 'median']).round(3))
method_boxplot(
    orientation_events, 'BannerDelayTime',
    'Landing-to-confirmation latency', 'Seconds'
)
method_boxplot(
    orientation_events, 'HeadRotation3D',
    'Head rotation during landing-to-confirmation', 'Degrees'
)


BannerDelayTime: Shapiro–Wilk p=0.0000, Q–Q r=0.976, skew=1.019; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
HeadRotation3D: Shapiro–Wilk p=0.0000, Q–Q r=0.976, skew=-0.826; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
         BannerDelayTime                ... HeadRotation3D                
                   count   mean    std  ...           mean     std  median
Method                                  ...                               
baseline             732  2.032  1.004  ...          6.216   5.867   5.682
map                  743  3.187  2.261  ...         35.071  29.388  27.876
tsunami              756  2.215  1.206  ...          6.702   5.227   5.711

[3 rows x 8 columns]


Tsunami_quantitative_analysis.ipynb:code-cell-5:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### H2.2 — Perceived spatial awareness

**H2.2: Participants will report higher spatial-awareness ratings with Tsunami than with Minimap-assisted teleportation.**

**Method.** The composite averages position awareness, direction awareness, and reverse-scored reorientation effort. Paired differences are evaluated with a one-sided Wilcoxon signed-rank test.

**Interpretation.** A positive Tsunami–Minimap difference indicates better subjective spatial awareness with Tsunami. Statistical significance, confidence-interval direction, and effect size should be considered together.


In [19]:
rows = [paired_primary_result(spatial, 'SpatialAwarenessComposite', 'tsunami', 'map',
                      'greater', 'H2.2', 'primary')]
show_results('H2.2', rows)
display(spatial.groupby('Method')[['position_awareness', 'direction_awareness',
                                   'reorientation_effort_reversed', 'SpatialAwarenessComposite']]
        .agg(['mean', 'std']).round(3))
method_boxplot(spatial, 'SpatialAwarenessComposite',
               'Subjective spatial awareness', 'Score (1–6)')


def cronbach_alpha_items(frame):
    values = frame.dropna().to_numpy(float)
    item_variances = values.var(axis=0, ddof=1)
    total_variance = values.sum(axis=1).var(ddof=1)
    k = values.shape[1]
    return k / (k - 1) * (1 - item_variances.sum() / total_variance)

h22_reliability_rows = []
for method, method_data in spatial.groupby('Method'):
    items = method_data[[
        'position_awareness', 'direction_awareness',
        'reorientation_effort_reversed',
    ]]
    h22_reliability_rows.append({
        'Method': method, 'N': len(items.dropna()),
        'CronbachAlpha': cronbach_alpha_items(items),
    })
h22_reliability = pd.DataFrame(h22_reliability_rows)
RESULTS['H2.2_composite_reliability'] = h22_reliability
display(h22_reliability.style.format(precision=3))


diverging_likert_plot(
    spatial,
    [
        ('position_awareness', 'Position awareness'),
        ('direction_awareness', 'Direction awareness'),
        ('reorientation_effort', 'Additional reorientation effort'),
        ('environmental_continuity', 'Environmental continuity'),
    ],
    'Spatial-awareness item distributions',
)


         position_awareness         ... SpatialAwarenessComposite       
                       mean    std  ...                      mean    std
Method                              ...                                 
baseline              5.024  1.334  ...                     5.016  0.988
map                   2.857  1.507  ...                     3.056  1.094
tsunami               4.548  1.468  ...                     4.619  1.211

[3 rows x 8 columns]


Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:386: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Environmental continuity (exploratory)

**Method.** Environmental continuity is retained as an exploratory RQ2 outcome, not as a separate hypothesis. Participant-level Tsunami–Minimap ratings are compared with a two-sided Wilcoxon signed-rank test.

**Interpretation.** The contrast describes whether perceived environmental continuity differed between the two long-range techniques. It may contextualize H2.2, but it is not hypothesis-directed evidence for an additional hypothesis.


In [20]:
rows = [paired_primary_result(
    spatial, 'environmental_continuity', 'tsunami', 'map', 'two-sided',
    'RQ2', 'exploratory perceived environmental continuity',
)]
show_results('RQ2_environmental_continuity', rows)
method_boxplot(
    spatial, 'environmental_continuity',
    'Perceived environmental continuity', 'Score (1–6)',
)


Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## RQ2 — Exploratory analyses


### RQ2 exploratory — Post-confirmation aiming behaviour

**Question.** How do participants initiate the next locomotion interaction after confirming the reached checkpoint, and does this behavior change over repeated use?

**Method.** `PostBannerAimLatency` and `HeadRotationBannerToNextAim3D` use the interval `progress_banner → next method-specific aim`. This phase starts only after checkpoint confirmation makes the next destination available. Two-sided event-level log1p LMMs compare the methods, and participant-level method means provide sensitivity summaries. A separate longitudinal model estimates within-method progression and Method × Progress. These analyses are exploratory and are outside every hypothesis-directed Holm family.

**Interpretation.** The outcomes measure initiation of the next locomotion interaction, not H2.1 landing-to-checkpoint reorientation. Minimap users can return immediately to the controller-mounted map and begin the next map-based aim, whereas Tsunami users often orient their world-space interaction toward the next destination. Consequently, smaller post-confirmation latency or head rotation with Minimap must not be interpreted as lower post-teleport reorientation effort. Head/controller geometry, posture, and target layout can also affect these measures.


In [21]:
ANALYSIS_CONTEXT = {'Hypothesis': 'RQ2', 'Analysis': 'RQ2 exploratory — Post-confirmation aiming behaviour'}
post_confirmation_rows = []
post_confirmation_residuals = {}
for outcome, role in (
    ('PostBannerAimLatency', 'exploratory post-confirmation latency'),
    ('HeadRotationBannerToNextAim3D', 'exploratory post-confirmation rotation'),
):
    result, residuals = h21_event_lmm(
        orientation_events, outcome, role, alternative='two-sided'
    )
    result['Hypothesis'] = 'RQ2 exploratory'
    post_confirmation_rows.append(result)
    post_confirmation_residuals[outcome] = residuals
post_confirmation_lmm_table = pd.DataFrame(post_confirmation_rows)
post_confirmation_lmm_table['PHolm'] = holm(
    post_confirmation_lmm_table['PValue'].to_numpy()
)
RESULTS['RQ2_exploratory_post_confirmation_aiming_LMM'] = (
    post_confirmation_lmm_table
)
display(post_confirmation_lmm_table.style.format(precision=4))
residual_diagnostic_plot(
    post_confirmation_residuals,
    'RQ2 exploratory — post-confirmation aiming LMM diagnostics',
)

post_confirmation_participant_rows = []
for outcome in ('PostBannerAimLatency', 'HeadRotationBannerToNextAim3D'):
    participant_means = aggregate_method(orientation_events, outcome)
    post_confirmation_participant_rows.append(paired_primary_result(
        participant_means, outcome, 'tsunami', 'map', 'two-sided',
        'RQ2', 'exploratory post-confirmation aiming behaviour',
    ))
show_results(
    'RQ2_exploratory_post_confirmation_participant',
    post_confirmation_participant_rows, adjust=True,
)
display(orientation_events.groupby('Method')[[
    'PostBannerAimLatency', 'HeadRotationBannerToNextAim3D',
]].agg(['count', 'mean', 'std', 'median']).round(3))
method_boxplot(
    orientation_events, 'PostBannerAimLatency',
    'Post-confirmation aiming latency', 'Seconds',
)
method_boxplot(
    orientation_events, 'HeadRotationBannerToNextAim3D',
    'Post-confirmation aiming head rotation', 'Degrees',
)

post_banner_trend_table = pd.DataFrame()
if POST_CONFIRMATION_AIMING_AVAILABLE:
    post_banner = orientation_events.loc[
        orientation_events['Method'].isin(['map', 'tsunami'])
    ].dropna(subset=[
        'PostBannerAimLatency', 'HeadRotationBannerToNextAim3D',
        'NextAimTimestamp', 'ParticipantUID', 'Method', 'Route',
        'MethodOrder', 'Location',
    ]).copy()
    post_banner = post_banner.sort_values(
        ['ParticipantUID', 'Method', 'NextAimTimestamp']
    )
    post_banner['EventOrderWithinMethod'] = (
        post_banner.groupby(['ParticipantUID', 'Method']).cumcount() + 1
    )
    post_banner['EligibleEventsWithinMethod'] = post_banner.groupby(
        ['ParticipantUID', 'Method']
    )['EventOrderWithinMethod'].transform('max')
    post_banner['WithinMethodProgress'] = np.where(
        post_banner['EligibleEventsWithinMethod'] > 1,
        (post_banner['EventOrderWithinMethod'] - 1)
        / (post_banner['EligibleEventsWithinMethod'] - 1),
        0.0,
    )
    post_banner['ProgressThird'] = pd.cut(
        post_banner['WithinMethodProgress'],
        bins=[-0.001, 1/3, 2/3, 1.001],
        labels=['early', 'middle', 'late'],
    )

    trend_rows = []
    for outcome in ('PostBannerAimLatency', 'HeadRotationBannerToNextAim3D'):
        model_data = post_banner.dropna(subset=[outcome]).copy()
        model_data['ModelOutcome'] = np.log1p(model_data[outcome])
        model = smf.mixedlm(
            'ModelOutcome ~ C(Method, Treatment(reference="map")) * '
            'WithinMethodProgress + C(Route) + C(MethodOrder) + C(Location)',
            model_data, groups=model_data['ParticipantUID'], re_formula='1',
        ).fit(reml=False, method='powell', maxiter=5000, disp=False)
        progress_name = 'WithinMethodProgress'
        interaction_name = (
            'C(Method, Treatment(reference="map"))[T.tsunami]:'
            'WithinMethodProgress'
        )
        map_slope = float(model.params[progress_name])
        tsunami_slope = map_slope + float(model.params[interaction_name])
        covariance = model.cov_params()
        tsunami_slope_se = float(np.sqrt(
            covariance.loc[progress_name, progress_name]
            + covariance.loc[interaction_name, interaction_name]
            + 2 * covariance.loc[progress_name, interaction_name]
        ))
        for method, slope, slope_se in (
            ('map', map_slope, float(model.bse[progress_name])),
            ('tsunami', tsunami_slope, tsunami_slope_se),
        ):
            z_value = slope / slope_se
            slope_ci = slope + np.array([-1, 1]) * stats.norm.ppf(0.975) * slope_se
            trend_rows.append({
                'Outcome': outcome, 'Effect': f'{method} temporal slope',
                'NParticipants': model_data['ParticipantUID'].nunique(),
                'NEvents': len(model_data), 'Log1pSlope': slope,
                'EndToStartOutcomePlusOneRatio': np.exp(slope),
                'RatioCI95Low': np.exp(slope_ci[0]),
                'RatioCI95High': np.exp(slope_ci[1]),
                'ZValue': z_value,
                'PValueTwoSided': 2 * stats.norm.sf(abs(z_value)),
                'Converged': bool(model.converged),
            })
        interaction = float(model.params[interaction_name])
        interaction_se = float(model.bse[interaction_name])
        interaction_z = interaction / interaction_se
        interaction_ci = interaction + np.array([-1, 1]) * stats.norm.ppf(0.975) * interaction_se
        trend_rows.append({
            'Outcome': outcome, 'Effect': 'Tsunami × progress interaction',
            'NParticipants': model_data['ParticipantUID'].nunique(),
            'NEvents': len(model_data), 'Log1pSlope': interaction,
            'EndToStartOutcomePlusOneRatio': np.exp(interaction),
            'RatioCI95Low': np.exp(interaction_ci[0]),
            'RatioCI95High': np.exp(interaction_ci[1]),
            'ZValue': interaction_z,
            'PValueTwoSided': 2 * stats.norm.sf(abs(interaction_z)),
            'Converged': bool(model.converged),
        })
    post_banner_trend_table = pd.DataFrame(trend_rows)
    RESULTS['RQ2_post_banner_longitudinal_LMM'] = post_banner_trend_table
    display(post_banner_trend_table.style.format(precision=4))
    display(post_banner.groupby(['Method', 'ProgressThird'], observed=True)[
        ['PostBannerAimLatency', 'HeadRotationBannerToNextAim3D']
    ].agg(['count', 'mean', 'std', 'median']).round(3))

    for outcome, ylabel, title in (
        ('PostBannerAimLatency', 'Seconds', 'Post-banner latency over repeated use'),
        ('HeadRotationBannerToNextAim3D', 'Degrees', 'Post-banner 3D head rotation over repeated use'),
    ):
        fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
        for ax, method in zip(axes, ('map', 'tsunami')):
            subset = post_banner.loc[post_banner['Method'].eq(method)]
            for _, participant_data in subset.groupby('ParticipantUID'):
                ax.plot(
                    participant_data['WithinMethodProgress'], participant_data[outcome],
                    color=METHOD_COLORS[method], alpha=0.18, linewidth=0.8,
                )
            sns.regplot(
                data=subset, x='WithinMethodProgress', y=outcome,
                scatter=False, lowess=True, ax=ax,
                color=METHOD_COLORS[method], line_kws={'linewidth': 3},
            )
            ax.set_title(METHOD_LABELS[method])
            ax.set_xlabel('Progress within method (0 = first, 1 = last)')
            ax.set_ylabel(ylabel)
        fig.suptitle(title)
        fig.tight_layout()
        plt.show()
else:
    print(
        'Post-banner longitudinal analysis skipped: regenerate '
        'city_race_orientation_events.csv with the current City Race '
        'preprocessing script to add the next-aim fields.'
    )


PostBannerAimLatency: Shapiro–Wilk p=0.0000, Q–Q r=0.977, skew=1.015; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
HeadRotationBannerToNextAim3D: Shapiro–Wilk p=0.0000, Q–Q r=0.970, skew=-1.051; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
         PostBannerAimLatency                ... HeadRotationBannerToNextAim3D                
                        count   mean    std  ...                          mean     std  median
Method                                       ...                                              
baseline                  732  1.480  0.788  ...                        66.150  37.602  66.166
map                       742  1.265  1.111  ...                        42.084  36.086  30.799
tsunami                   756  1.399  0.805  ...                        70.378  42.536  68.255

[3 rows x 8 columns]
                      PostBannerAimLatency         ... HeadRotationBannerToNex

Tsunami_quantitative_analysis.ipynb:code-cell-5:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-21:163: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-21:145: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
Tsunami_quantitative_analysis.ipynb:code-cell-21:163: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be sh

### H2.1 supplementary — Yaw/pitch decomposition of landing-to-confirmation head rotation

**Method.** `AbsoluteYawChange` and `AbsolutePitchChange` decompose hypothesis-directed H2.1 `HeadRotation3D` over the same `appear → progress_banner` interval. Separate two-sided log1p LMMs use the H2.1 covariates and participant random intercept; Holm correction is applied to this two-test supplementary family only.

**Interpretation.** The decomposition shows whether the aggregate 3D movement difference is present in horizontal and/or vertical viewing-direction changes. It strengthens interpretation of the physical movement pattern but neither independently tests nor redefines H2.1, and head movement alone does not prove environmental awareness.


In [22]:
ANALYSIS_CONTEXT = {'Hypothesis': 'H2.1', 'Analysis': 'H2.1 supplementary — Yaw/pitch decomposition of landing-to-confirmation head rotation'}
def h21_yaw_pitch_lmm(frame, outcome):
    model_data = frame.dropna(subset=[
        outcome, 'ParticipantUID', 'Method', 'TargetID', 'MethodOrder', 'Location'
    ]).copy()
    model_data['ModelOutcome'] = np.log1p(model_data[outcome])
    formula = (
        'ModelOutcome ~ C(Method, Treatment(reference="map")) + '
        'C(TargetID) + C(MethodOrder) + C(Location)'
    )
    model = smf.mixedlm(
        formula, model_data, groups=model_data['ParticipantUID'], re_formula='1'
    ).fit(reml=False, method='powell', maxiter=5000, disp=False)
    parameter = 'C(Method, Treatment(reference="map"))[T.tsunami]'
    beta = float(model.params[parameter])
    se = float(model.bse[parameter])
    z_value = beta / se
    ci = beta + np.array([-1, 1]) * stats.norm.ppf(0.975) * se
    result = {
        'Outcome': outcome, 'Contrast': 'Tsunami - Minimap',
        'Alternative': 'two-sided',
        'NParticipants': model_data['ParticipantUID'].nunique(),
        'NEvents': len(model_data), 'Log1pDifference': beta,
        'AdjustedOutcomePlusOneRatio': np.exp(beta),
        'RatioCI95Low': np.exp(ci[0]), 'RatioCI95High': np.exp(ci[1]),
        'ZValue': z_value, 'PValue': 2 * stats.norm.sf(abs(z_value)),
        'Converged': bool(model.converged),
        'RandomInterceptVariance': float(model.cov_re.iloc[0, 0]),
    }
    return result, np.asarray(model.resid)

orientation_model_rows = []
orientation_residuals = {}
for outcome in ('AbsoluteYawChange', 'AbsolutePitchChange'):
    result, residuals = h21_yaw_pitch_lmm(orientation_events, outcome)
    orientation_model_rows.append(result)
    orientation_residuals[outcome] = residuals
orientation_behaviour_lmm = pd.DataFrame(orientation_model_rows)
orientation_behaviour_lmm['PHolm'] = holm(orientation_behaviour_lmm['PValue'])
RESULTS['H2.1_supplementary_yaw_pitch_decomposition'] = orientation_behaviour_lmm
display(orientation_behaviour_lmm.style.format(precision=4))
orientation_behaviour_diagnostics = residual_diagnostic_plot(
    orientation_residuals,
    'H2.1 supplementary — yaw/pitch LMM residual diagnostics',
)
display(orientation_events.groupby('Method')[
    ['BannerDelayTime', 'HeadRotation3D', 'AbsoluteYawChange',
     'AbsolutePitchChange', 'SignedPitchChange',
     'DirectionalErrorAtProgressBanner3D',
     'DirectionalErrorReduction3D']
].agg(['count', 'mean', 'std', 'median']).round(3))
method_boxplot(
    orientation_events, 'AbsoluteYawChange',
    'Horizontal head-direction change before confirmation', 'Absolute yaw change (degrees)'
)
method_boxplot(
    orientation_events, 'AbsolutePitchChange',
    'Vertical head-direction change before confirmation', 'Absolute pitch change (degrees)'
)


AbsoluteYawChange: Shapiro–Wilk p=0.0000, Q–Q r=0.989, skew=0.291; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
AbsolutePitchChange: Shapiro–Wilk p=0.0000, Q–Q r=0.985, skew=-0.283; residual-normality concerns. Interpret the transformed LMM together with this diagnostic.
         BannerDelayTime                ... DirectionalErrorReduction3D               
                   count   mean    std  ...                        mean     std median
Method                                  ...                                           
baseline             732  2.032  1.004  ...                       0.519   5.878  0.000
map                  743  3.187  2.261  ...                       0.613  29.835  0.899
tsunami              756  2.215  1.206  ...                       0.492   5.507  0.000

[3 rows x 28 columns]


Tsunami_quantitative_analysis.ipynb:code-cell-5:237: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### RQ2 exploratory — Controller-relative Minimap mechanism evidence

**Method.** Controller-relative angles at `appear` and `progress_banner`, their within-event shift, and 15°, 20°, and 25° threshold descriptions characterize Minimap viewing geometry. Questionnaire associations are descriptive. All measures are exploratory.

**Interpretation.** The pattern may be consistent with a transition from the controller-centred minimap back to the world-space checkpoint or environment. It does not prove that participants ignored the environment, were less immersed, or had worse spatial cognition. Head/controller geometry, posture, and interaction technique provide alternative explanations.


In [23]:
map_orientation = orientation_events.loc[
    orientation_events['Method'].eq('map')
].copy()
attention_columns = [
    f'LookingAtNonDominantController{threshold}' for threshold in (15, 20, 25)
]
map_attention_by_participant = map_orientation.groupby(
    'ParticipantUID', as_index=False
).agg({
    **{column: 'mean' for column in attention_columns},
    'AppearHeadToNonDominantControllerAngle3D': 'mean',
    'NonDominantControllerAttentionShift': 'mean',
    'AbsoluteYawChange': 'mean',
    'AbsolutePitchChange': 'mean',
})
map_questionnaire = spatial.loc[
    spatial['Method'].eq('map'),
    ['ParticipantUID', 'SpatialAwarenessComposite', 'environmental_continuity'],
]
map_attention_by_participant = map_attention_by_participant.merge(
    map_questionnaire, on='ParticipantUID', validate='one_to_one'
)

attention_sensitivity = pd.DataFrame([
    {
        'ViewingThresholdDegrees': threshold,
        'MeanParticipantAttentionRate': map_attention_by_participant[
            f'LookingAtNonDominantController{threshold}'
        ].mean(),
        'MedianParticipantAttentionRate': map_attention_by_participant[
            f'LookingAtNonDominantController{threshold}'
        ].median(),
    }
    for threshold in (15, 20, 25)
])
display(attention_sensitivity.style.format({
    'MeanParticipantAttentionRate': '{:.1%}',
    'MedianParticipantAttentionRate': '{:.1%}',
}))

map_controller_transition_summary = pd.DataFrame([{
    'NEvents': len(map_orientation),
    'MeanAngleToControllerAtAppear': map_orientation[
        'AppearHeadToNonDominantControllerAngle3D'
    ].mean(),
    'MeanAngleToControllerAtBanner': map_orientation[
        'ProgressBannerHeadToNonDominantControllerAngle3D'
    ].mean(),
    'MeanAttentionShift': map_orientation[
        'NonDominantControllerAttentionShift'
    ].mean(),
    'MedianAttentionShift': map_orientation[
        'NonDominantControllerAttentionShift'
    ].median(),
    'MeanSignedPitchChange': map_orientation['SignedPitchChange'].mean(),
    'MedianSignedPitchChange': map_orientation['SignedPitchChange'].median(),
}])
display(map_controller_transition_summary.style.format(precision=3))

participant_shift = map_attention_by_participant[
    'NonDominantControllerAttentionShift'
].dropna().to_numpy()
attention_registry_key = (
    'RQ2 exploratory', 'controller-attention transition',
    'NonDominantControllerAttentionShift', 'progress_banner', 'appear', 'greater'
)
if not any(row['RegistryKey'] == attention_registry_key for row in PAIRED_COMPARISONS):
    PAIRED_COMPARISONS.append({
        'RegistryKey': attention_registry_key,
        'Hypothesis': 'RQ2 exploratory',
        'Analysis': 'controller-attention transition',
        'Metric': 'NonDominantControllerAttentionShift',
        'First': 'progress_banner', 'Second': 'appear',
        'Contrast': 'Progress banner - Appear',
        'Alternative': 'greater',
        'Differences': participant_shift.copy(),
    })
shift_test = stats.wilcoxon(
    participant_shift, alternative='greater', zero_method='wilcox',
    correction=False, method='auto',
)
shift_method = 'One-sided Wilcoxon signed-rank test on participant means'
attention_registered = next(
    row for row in PAIRED_COMPARISONS if row['RegistryKey'] == attention_registry_key
)
attention_registered['CurrentSelectedMethod'] = shift_method
attention_registered['CurrentSelectedPRaw'] = float(shift_test.pvalue)
map_attention_shift_test = pd.DataFrame([{
    'Outcome': 'NonDominantControllerAttentionShift',
    'Alternative': 'greater than zero',
    'NParticipants': len(participant_shift),
    'MeanParticipantShift': participant_shift.mean(),
    'MedianParticipantShift': np.median(participant_shift),
    'ShapiroW': np.nan,
    'ShapiroP': np.nan,
    'TestStatistic': shift_test.statistic,
    'PValue': shift_test.pvalue,
    'TestMethod': shift_method,
    'Interpretation': 'Exploratory controller-to-banner attention transition',
}])
RESULTS['RQ2_exploratory_minimap_attention_shift'] = map_attention_shift_test
display(map_attention_shift_test.style.format(precision=4))

attention_associations = []
for threshold in (15, 20, 25):
    attention = map_attention_by_participant[
        f'LookingAtNonDominantController{threshold}'
    ]
    for rating in ('SpatialAwarenessComposite', 'environmental_continuity'):
        rho, p_value = stats.spearmanr(
            attention, map_attention_by_participant[rating], nan_policy='omit'
        )
        attention_associations.append({
            'ViewingThresholdDegrees': threshold,
            'QuestionnaireOutcome': rating,
            'SpearmanRho': rho,
            'PValue': p_value,
            'NParticipants': len(map_attention_by_participant),
            'Interpretation': 'Exploratory association; no causal interpretation',
        })
attention_associations = pd.DataFrame(attention_associations)
attention_associations['PHolm'] = holm(attention_associations['PValue'])
display(attention_associations.style.format(precision=4))

display(city.groupby('Method')[['MeanOrientationTime', 'MeanHeadDelta',
                                'MeanReorientationImprovement',
                                'MeanNearestControllerViewingAngle',
                                'ControllerDirectedAttentionRate']].mean().round(3))
display(spatial.groupby('Method')[['environmental_continuity',
                                   'DistanceAwarenessComposite']].mean().round(3))


          MeanOrientationTime  ...  ControllerDirectedAttentionRate
Method                         ...                                 
baseline                1.082  ...                            0.002
map                     4.253  ...                            0.068
tsunami                 2.225  ...                            0.011

[3 rows x 5 columns]
          environmental_continuity  DistanceAwarenessComposite
Method                                                        
baseline                     4.381                       3.917
map                          2.905                       3.738
tsunami                      4.405                       4.381


## RQ3 — Usability Trade-Offs

What usability and comfort trade-offs emerge for Tsunami relative to Minimap-assisted and conventional teleportation?

We anticipated that visible world deformation could increase cybersickness relative to Baseline, whereas avoiding a separate map-selection interface could reduce workload relative to Minimap. We therefore formulated two directional hypotheses:


### H3.1 — Cybersickness symptom change

**H3.1: Tsunami will produce greater cybersickness symptom increases than Baseline teleportation.**

For each method, `CSQTotalChange` is the post-condition total minus the immediately preceding CSQ-VR total according to counterbalanced condition order. The hypothesis-directed contrast is Tsunami minus Baseline with the one-sided alternative `greater`. Because this family contains one test, its Holm-adjusted value equals the raw p-value. Comparisons involving Minimap are exploratory and two-sided.

The contrasts apply to brief within-session exposures. Because post-condition measurements followed accumulating VR exposure, residual carry-over cannot be excluded despite counterbalancing. Non-significant contrasts do not establish equivalent comfort or safety.


In [24]:
h31_rows = [paired_primary_result(
    csq_long, 'CSQTotalChange', 'tsunami', 'baseline', 'greater',
    'H3.1', 'hypothesis-directed',
)]
h31_table = show_results('H3.1_primary', h31_rows)

h31_minimap_rows = [
    paired_primary_result(
        csq_long, 'CSQTotalChange', 'map', 'baseline', 'two-sided',
        'RQ3', 'exploratory CSQ-VR: Minimap versus Baseline',
    ),
    paired_primary_result(
        csq_long, 'CSQTotalChange', 'map', 'tsunami', 'two-sided',
        'RQ3', 'exploratory CSQ-VR: Minimap versus Tsunami',
    ),
]
h31_minimap_table = show_results(
    'RQ3_exploratory_CSQ_minimap', h31_minimap_rows, adjust=True,
)

display(csq_long.groupby('Method')[[
    'CSQTotalBefore', 'CSQTotalAfter', 'CSQTotalChange'
]].agg(['count', 'mean', 'std', 'median']).round(3))
method_boxplot(
    csq_long, 'CSQTotalChange',
    'Within-session CSQ-VR total change', 'CSQ-VR change',
)


         CSQTotalBefore                ... CSQTotalChange              
                  count   mean    std  ...           mean    std median
Method                                 ...                             
baseline             42  7.548  2.432  ...          0.333  1.959    0.0
map                  42  7.619  2.263  ...          0.167  1.480    0.0
tsunami              42  7.214  1.570  ...          0.286  1.729    0.0

[3 rows x 12 columns]


Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### H3.2 — Perceived workload

**H3.2: Participants will report lower perceived workload with Tsunami than with Minimap-assisted teleportation.**

**Method.** Participant-level Tsunami–Minimap RAW-TLX differences are tested with Wilcoxon signed ranks. The pre-specified primary test is a one-sided Wilcoxon signed-rank test. Raw differences, confidence intervals, and rank-biserial effects accompany the contrast.

**Interpretation.** A negative Tsunami–Minimap difference supports H3.2. If the one-sided p-value is not below `ALPHA`, the data do not provide sufficient evidence that Tsunami reduces workload; this does not demonstrate equality between the methods.


In [25]:
rows = [paired_primary_result(tlx_long, 'RAWTLX', 'tsunami', 'map', 'less',
                      'H3.2', 'primary')]
show_results('H3.2', rows)
display(tlx_long.groupby('Method')['RAWTLX'].agg(['mean', 'std', 'median']).round(3))
method_boxplot(tlx_long, 'RAWTLX', 'RAW-TLX perceived workload', 'Score (0–100)')


            mean     std  median
Method                          
baseline  20.377  12.666  16.670
map       21.151  12.517  19.585
tsunami   17.778  10.764  16.670


Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## RQ3 — Exploratory and contextual analyses


### RQ3 contextual — CSQ-VR development over experimental time

**Method.** CSQ-VR Total is shown descriptively at the pre-experiment measurement and after method blocks 1, 2, and 3, irrespective of which counterbalanced method occurred in a given block. Thin lines show individual participants; the thick line shows the sample mean. A companion boxplot summarizes the distribution at each measurement occasion. No inferential test is performed in this cell.

**Interpretation.** A broadly flat trajectory suggests little cumulative symptom drift over the experiment, whereas a systematic upward trajectory suggests accumulating cybersickness or fatigue. This temporal plot cannot attribute a change to a specific locomotion method because method order is counterbalanced. Absence of a visually large change is contextual evidence only and does not prove that methods had no effect; the method-specific change analysis in H3.1 remains the relevant inferential test.


In [26]:
measurement_order = [0, 1, 2, 3]
measurement_labels = [
    'Pre-experiment', 'After block 1', 'After block 2', 'After block 3'
]
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), sharey=True)

wide_time = csq_time.pivot(
    index='ParticipantUID', columns='Measurement', values='CSQTotal'
).reindex(columns=measurement_order)
for _, participant_values in wide_time.iterrows():
    axes[0].plot(
        measurement_order, participant_values.to_numpy(float),
        color='0.72', linewidth=0.9, alpha=0.65,
    )
mean_time = wide_time.mean(axis=0)
median_time = wide_time.median(axis=0)
axes[0].plot(measurement_order, mean_time, marker='o', linewidth=3,
             color='#2166ac', label='Mean')
axes[0].plot(measurement_order, median_time, marker='s', linewidth=2,
             linestyle='--', color='#b2182b', label='Median')
axes[0].set_xticks(measurement_order)
axes[0].set_xticklabels(measurement_labels, rotation=18, ha='right')
axes[0].set_xlabel('Measurement occasion')
axes[0].set_ylabel('CSQ-VR Total')
axes[0].set_title('Individual and aggregate trajectories')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.34), ncol=2)
axes[0].grid(alpha=0.25)

sns.boxplot(
    data=csq_time, x='Measurement', y='CSQTotal',
    order=measurement_order, color='#9ecae1', showfliers=False, ax=axes[1],
)
sns.stripplot(
    data=csq_time, x='Measurement', y='CSQTotal',
    order=measurement_order, color='black', alpha=0.45,
    jitter=0.16, size=3.5, ax=axes[1],
)
axes[1].set_xticks(range(len(measurement_order)))
axes[1].set_xticklabels(measurement_labels, rotation=18, ha='right')
axes[1].set_xlabel('Measurement occasion')
axes[1].set_ylabel('')
axes[1].set_title('Distribution at each occasion')
axes[1].grid(axis='y', alpha=0.25)

fig.suptitle('CSQ-VR Total over the course of the experiment', y=1.02)
fig.tight_layout(rect=(0, 0.24, 1, 1))
plt.show()

display(
    csq_time.groupby(['Measurement', 'MeasurementLabel'])['CSQTotal']
    .agg(['count', 'mean', 'std', 'median', 'min', 'max']).round(3)
)


                                  count   mean    std  median  min   max
Measurement MeasurementLabel                                            
0           Pre-experiment           42  7.000  1.530     6.0  6.0  11.0
1           After method block 1     42  7.643  2.184     6.5  6.0  14.0
2           After method block 2     42  7.738  2.480     7.0  6.0  16.0
3           After method block 3     42  7.786  2.312     7.0  6.0  14.0


Tsunami_quantitative_analysis.ipynb:code-cell-26:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


### Subjective usability, distance awareness, and rankings

**Method.** two-sided Wilcoxon signed-rank tests compare Tsunami with Minimap for usability and distance-awareness composites. For each complete-ranking preference question, a Friedman test evaluates the overall within-participant method difference and Kendall's W quantifies its magnitude. Pairwise Wilcoxon tests with Holm correction localize significant omnibus results. Usability items and preferences are visualized with diverging Likert and 100% stacked ranking bars.

**Interpretation.** Positive composite differences favour Tsunami. In the Likert chart, a larger right-hand share indicates more agreement; in the ranking chart, a larger first-place share indicates stronger preference. Because these outcomes are exploratory, p-values and visual patterns should be interpreted as evidence-generating rather than hypothesis-directed.


In [27]:
rows = [
    paired_primary_result(spatial, 'UsabilityComposite', 'tsunami', 'map', 'two-sided', 'RQ3', 'exploratory'),
    paired_primary_result(spatial, 'DistanceAwarenessComposite', 'tsunami', 'map', 'two-sided', 'RQ3', 'exploratory'),
]
show_results('RQ3', rows)

preferences = raw['preferences']
preferences = preferences.copy()
preferences['ParticipantUID'] = participant_uid(
    preferences['Location'], preferences['Participant ID']
)
ranking_prefixes = ['Overall Preference for Long-Range Navigation', 'Comfort During Use',
                    'Perceived Intuitiveness', 'Fun Factor']
ranking_categories = ['1st', '2nd', '3rd']
ranking_rows = []
for prefix in ranking_prefixes:
    for method in METHODS:
        column = f'{prefix} [{METHOD_LABELS[method]}]'
        counts = preferences[column].value_counts().reindex(ranking_categories, fill_value=0)
        ranking_rows.append({
            'Question': prefix, 'Method': method,
            **{category: int(counts[category]) for category in ranking_categories},
        })
ranking_distribution = pd.DataFrame(ranking_rows)
display(ranking_distribution)

preference_omnibus_rows = []
preference_pairwise_rows = []
rank_value = {'1st': 1.0, '2nd': 2.0, '3rd': 3.0}
for prefix in ranking_prefixes:
    wide_ranks = pd.DataFrame({
        method: preferences[f'{prefix} [{METHOD_LABELS[method]}]'].map(rank_value)
        for method in METHODS
    })
    wide_ranks['ParticipantUID'] = preferences['ParticipantUID'].to_numpy()
    wide_ranks = wide_ranks.dropna(subset=list(METHODS))
    friedman = stats.friedmanchisquare(*[
        wide_ranks[method].to_numpy(float) for method in METHODS
    ])
    n_participants, n_methods = len(wide_ranks), len(METHODS)
    preference_omnibus_rows.append({
        'Question': prefix, 'Test': 'Friedman test',
        'NParticipants': n_participants, 'ChiSquare': friedman.statistic,
        'DegreesOfFreedom': n_methods - 1, 'PValue': friedman.pvalue,
        'KendallsW': friedman.statistic / (n_participants * (n_methods - 1)),
    })
    long_ranks = wide_ranks.melt(
        id_vars='ParticipantUID', value_vars=list(METHODS),
        var_name='Method', value_name='PreferenceRank',
    )
    question_rows = []
    for first, second in (
        ('baseline', 'map'), ('baseline', 'tsunami'), ('map', 'tsunami')
    ):
        question_rows.append(paired_wilcoxon_result(
            long_ranks, 'PreferenceRank', first, second, 'two-sided',
            'RQ3 preferences', prefix,
        ))
    question_table = pd.DataFrame(question_rows)
    question_table['PHolm'] = holm(question_table['PValue'])
    preference_pairwise_rows.extend(question_table.to_dict('records'))

preference_omnibus = pd.DataFrame(preference_omnibus_rows)
preference_omnibus['PHolmAcrossQuestions'] = holm(preference_omnibus['PValue'])
preference_pairwise = pd.DataFrame(preference_pairwise_rows)
RESULTS['RQ3_preference_omnibus'] = preference_omnibus
RESULTS['RQ3_preference_pairwise'] = preference_pairwise
display(preference_omnibus.style.format(precision=4))
display(preference_pairwise.style.format(precision=4))

diverging_likert_plot(
    spatial,
    [('easy_to_learn', 'Easy to learn'), ('intuitive', 'Intuitive to use')],
    'RQ3 usability-item response distributions',
)

ranking_colors = {'1st': '#0072B2', '2nd': '#E69F00', '3rd': '#CC79A7'}
fig, ax = plt.subplots(figsize=(12, 7.5))
ranking_plot = ranking_distribution.copy()
ranking_plot['Label'] = ranking_plot.apply(
    lambda row: f"{row['Question']} — {METHOD_LABELS[row['Method']]}", axis=1
)
rows_per_question = len(METHODS)
group_stride = rows_per_question + 1
y_positions = np.array([
    question * group_stride + method
    for question in range(len(ranking_prefixes))
    for method in range(rows_per_question)
], dtype=float)
left = np.zeros(len(ranking_plot))
totals = ranking_plot[ranking_categories].sum(axis=1).to_numpy(float)
for category in ranking_categories:
    values = ranking_plot[category].to_numpy(float)
    ax.barh(
        y_positions, values, left=left, label=category,
        color=ranking_colors[category], edgecolor='white', linewidth=0.5,
    )
    left += values
participant_count = int(totals.max())
ax.set_yticks(y_positions)
ax.set_yticklabels(ranking_plot['Label'])
ax.invert_yaxis()
ax.set_xlim(0, participant_count)
ax.set_xticks(np.arange(0, participant_count + 1, 5))
ax.set_xlabel('Number of participants')
ax.set_title('RQ3 preference-ranking distributions')
ax.legend(title='Rank', ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.20))
ax.grid(axis='x', alpha=0.2)
fig.tight_layout(rect=(0, 0.20, 1, 1))
plt.show()

method_boxplot(spatial, 'UsabilityComposite', 'Usability composite', 'Score (1–6)')


                                        Question    Method  1st  2nd  3rd
0   Overall Preference for Long-Range Navigation  baseline    5   17   20
1   Overall Preference for Long-Range Navigation       map   11   12   19
2   Overall Preference for Long-Range Navigation   tsunami   26   13    3
3                             Comfort During Use  baseline   13   19   10
4                             Comfort During Use       map    9    5   28
5                             Comfort During Use   tsunami   20   18    4
6                        Perceived Intuitiveness  baseline   21   17    4
7                        Perceived Intuitiveness       map    3    8   31
8                        Perceived Intuitiveness   tsunami   18   17    7
9                                     Fun Factor  baseline    9   17   16
10                                    Fun Factor       map    4   13   25
11                                    Fun Factor   tsunami   29   12    1


Tsunami_quantitative_analysis.ipynb:code-cell-5:386: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-27:110: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
Tsunami_quantitative_analysis.ipynb:code-cell-5:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Sensitivity analyses

**Purpose.** This section systematically evaluates whether all simple participant-level paired comparisons can use the Wilcoxon signed-rank test without materially changing inference relative to the paired t-test. Primary LMM analyses are not included or modified. Every row reuses the exact paired differences already supplied to a conventional paired comparison elsewhere in the notebook.

**Tests and estimands.** Both tests use the same alternative: one-sided only when the associated hypothesis specifies a direction, otherwise two-sided. Wilcoxon uses `zero_method='wilcox'`, so zero differences are excluded from the signed-rank statistic while remaining reported in `ZeroDifferences`; no continuity correction is applied and SciPy's automatic exact/asymptotic method selection is used. The table reports raw and Holm-adjusted p-values, raw-unit mean and median differences, a 95% t interval for the mean difference, Cohen's dz, and signed matched-pairs rank-biserial correlation.

**Interpretation.** This is not an observed-power analysis. Robustness is judged from agreement of adjusted conclusions, effect magnitudes, confidence intervals, zero-difference prevalence, and cases where Wilcoxon is visibly less sensitive. Holm correction is applied separately but identically to the t-test and Wilcoxon p-values within explicitly defined logical families.


In [28]:
print(f'Sensitivity comparisons registered: {len(PAIRED_COMPARISONS)}')

def paired_sensitivity_holm_family(row):
    hypothesis = row['Hypothesis']
    analysis = row['Analysis']
    metric = row['Metric']
    if hypothesis == 'H1.1':
        return f'H1.1::{metric}'
    if hypothesis == 'RQ1':
        return f'RQ1 strategy contrast::{metric}'
    if hypothesis == 'Route post-hoc':
        return f'Route post-hoc::{metric}'
    if hypothesis == 'H2.1' and analysis in {
        'primary temporal RL participant-mean sensitivity',
        'supporting angular RA participant-mean sensitivity',
    }:
        return 'H2.1 participant-level RL/RA family'
    if hypothesis == 'H2.1' and analysis in {
        'exploratory RRC participant-mean comparison',
        'exploratory movement-based RC participant-mean comparison',
    }:
        return 'H2.1 exploratory alternative-correction family'
    if hypothesis == 'H3.1':
        return 'H3.1 CSQ-VR family'
    if hypothesis == 'RQ3':
        return 'RQ3 exploratory composites'
    if hypothesis == 'RQ3 preferences':
        return f'RQ3 preferences::{analysis}'
    return f'{hypothesis}::{analysis}::{metric}'

paired_sensitivity_rows = []
for comparison in PAIRED_COMPARISONS:
    diff = np.asarray(comparison['Differences'], dtype=float)
    diff = diff[np.isfinite(diff)]
    alternative = comparison['Alternative']
    t_result = stats.ttest_1samp(diff, 0, alternative=alternative)
    if np.allclose(diff, 0):
        wilcoxon_statistic, wilcoxon_p = 0.0, 1.0
    else:
        wilcoxon_result = stats.wilcoxon(
            diff, alternative=alternative, zero_method='wilcox',
            correction=False, method='auto',
        )
        wilcoxon_statistic = float(wilcoxon_result.statistic)
        wilcoxon_p = float(wilcoxon_result.pvalue)
    mean_ci = paired_mean_ci(diff)
    difference_sd = diff.std(ddof=1)
    effect_dz = (diff.mean() / difference_sd if difference_sd > 0 else np.nan)
    rank_biserial = paired_rank_biserial(diff)
    paired_sensitivity_rows.append({
        'HypothesisAnalysis': (
            f"{comparison['Hypothesis']} — {comparison['Analysis']}"
        ),
        'Hypothesis': comparison['Hypothesis'],
        'Analysis': comparison['Analysis'],
        'Metric': comparison['Metric'],
        'Contrast': comparison['Contrast'],
        'Alternative': alternative,
        'WilcoxonZeroMethod': 'wilcox (zero differences excluded)',
        'WilcoxonCorrection': False,
        'WilcoxonMethod': 'auto',
        'NPairs': len(diff),
        'NonzeroPairs': int(np.sum(~np.isclose(diff, 0))),
        'ZeroDifferences': int(np.sum(np.isclose(diff, 0))),
        'ZeroDifferenceRate': float(np.mean(np.isclose(diff, 0))),
        'MeanDifference': float(diff.mean()),
        'MedianDifference': float(np.median(diff)),
        'MeanDifferenceCI95Low': mean_ci[0],
        'MeanDifferenceCI95High': mean_ci[1],
        'PairedTStatistic': float(t_result.statistic),
        'PairedTPRaw': float(t_result.pvalue),
        'CohensDz': effect_dz,
        'WilcoxonStatistic': wilcoxon_statistic,
        'WilcoxonPRaw': wilcoxon_p,
        'RankBiserialCorrelation': rank_biserial,
        'HolmFamily': paired_sensitivity_holm_family(comparison),
        'CurrentSelectedMethod': comparison.get('CurrentSelectedMethod', ''),
        'CurrentSelectedPRaw': comparison.get('CurrentSelectedPRaw', np.nan),
    })

paired_test_sensitivity = pd.DataFrame(paired_sensitivity_rows)
paired_test_sensitivity['PairedTPHolm'] = paired_test_sensitivity.groupby(
    'HolmFamily'
)['PairedTPRaw'].transform(lambda values: holm(values.to_numpy()))
paired_test_sensitivity['WilcoxonPHolm'] = paired_test_sensitivity.groupby(
    'HolmFamily'
)['WilcoxonPRaw'].transform(lambda values: holm(values.to_numpy()))
paired_test_sensitivity['CurrentSelectedPHolm'] = paired_test_sensitivity.groupby(
    'HolmFamily'
)['CurrentSelectedPRaw'].transform(lambda values: holm(values.to_numpy()))
has_current_selection = (
    paired_test_sensitivity['CurrentSelectedMethod'].astype(str).str.strip().ne('')
    & np.isfinite(paired_test_sensitivity['CurrentSelectedPRaw'])
)
paired_test_sensitivity.loc[
    ~has_current_selection, 'CurrentSelectedPHolm'
] = np.nan
paired_test_sensitivity['UniformWilcoxonChangesCurrentConclusion'] = (
    has_current_selection
    & ((paired_test_sensitivity['CurrentSelectedPHolm'] < ALPHA)
       != (paired_test_sensitivity['WilcoxonPHolm'] < ALPHA))
)

def paired_conclusion_agreement(row):
    expected_sign = {'less': -1, 'greater': 1}.get(row['Alternative'])
    mean_sign = np.sign(row['MeanDifference'])
    rank_sign = np.sign(row['RankBiserialCorrelation'])
    if expected_sign is not None:
        direction_ok = (
            mean_sign in (0, expected_sign) and rank_sign in (0, expected_sign)
        )
    else:
        direction_ok = mean_sign == 0 or rank_sign == 0 or mean_sign == rank_sign
    if not direction_ok:
        return 'direction inconsistent'
    t_significant = row['PairedTPHolm'] < ALPHA
    wilcoxon_significant = row['WilcoxonPHolm'] < ALPHA
    if t_significant and wilcoxon_significant:
        return 'same significant conclusion'
    if t_significant:
        return 't-test significant only'
    if wilcoxon_significant:
        return 'Wilcoxon significant only'
    return 'both non-significant'

paired_test_sensitivity['AgreementOfConclusions'] = paired_test_sensitivity.apply(
    paired_conclusion_agreement, axis=1
)

def paired_problem_flags(row):
    flags = []
    if row['AgreementOfConclusions'] == 't-test significant only':
        flags.append('adjusted significance only for t-test')
    if row['AgreementOfConclusions'] == 'Wilcoxon significant only':
        flags.append('adjusted significance only for Wilcoxon')
    if min(abs(row['PairedTPHolm'] - ALPHA), abs(row['WilcoxonPHolm'] - ALPHA)) <= 0.02:
        flags.append('adjusted p-value near .05')
    if abs(row['CohensDz']) < 0.2 or abs(row['RankBiserialCorrelation']) < 0.1:
        flags.append('small effect')
    ci_width = row['MeanDifferenceCI95High'] - row['MeanDifferenceCI95Low']
    if (
        row['MeanDifferenceCI95Low'] <= 0 <= row['MeanDifferenceCI95High']
        or ci_width > 2 * max(abs(row['MeanDifference']), np.finfo(float).eps)
    ):
        flags.append('wide or inconclusive mean-difference CI')
    if row['ZeroDifferenceRate'] >= 0.20:
        flags.append('many zero differences')
    if (
        row['PairedTPHolm'] < ALPHA and row['WilcoxonPHolm'] >= 0.10
    ):
        flags.append('material Wilcoxon sensitivity loss')
    if row['AgreementOfConclusions'] == 'direction inconsistent':
        flags.append('direction inconsistent')
    return '; '.join(flags)

paired_test_sensitivity['ProblemFlags'] = paired_test_sensitivity.apply(
    paired_problem_flags, axis=1
)
paired_test_sensitivity['Problematic'] = paired_test_sensitivity['ProblemFlags'].ne('')

summary_columns = [
    'HypothesisAnalysis', 'Metric', 'Contrast', 'Alternative', 'NPairs',
    'MeanDifference', 'MedianDifference', 'MeanDifferenceCI95Low',
    'MeanDifferenceCI95High', 'PairedTStatistic', 'PairedTPRaw',
    'PairedTPHolm', 'CohensDz', 'WilcoxonStatistic', 'WilcoxonPRaw',
    'WilcoxonPHolm', 'RankBiserialCorrelation', 'ZeroDifferences',
    'ZeroDifferenceRate', 'CurrentSelectedMethod', 'CurrentSelectedPHolm',
    'UniformWilcoxonChangesCurrentConclusion', 'AgreementOfConclusions',
    'ProblemFlags',
]
display(
    paired_test_sensitivity[summary_columns].style
    .format(precision=4)
    .apply(
        lambda row: [
            'background-color: #ffe0e0' if row['ProblemFlags'] else ''
            for _ in row
        ],
        axis=1,
    )
)

agreement_counts = paired_test_sensitivity['AgreementOfConclusions'].value_counts()
display(agreement_counts.rename('Comparisons').to_frame())
t_only = paired_test_sensitivity.loc[
    paired_test_sensitivity['AgreementOfConclusions'].eq('t-test significant only'),
    ['HypothesisAnalysis', 'Metric', 'Contrast', 'PairedTPHolm', 'WilcoxonPHolm'],
]
wilcoxon_only = paired_test_sensitivity.loc[
    paired_test_sensitivity['AgreementOfConclusions'].eq('Wilcoxon significant only'),
    ['HypothesisAnalysis', 'Metric', 'Contrast', 'PairedTPHolm', 'WilcoxonPHolm'],
]
direction_issues = paired_test_sensitivity.loc[
    paired_test_sensitivity['AgreementOfConclusions'].eq('direction inconsistent'),
    ['HypothesisAnalysis', 'Metric', 'Contrast', 'MeanDifference',
     'RankBiserialCorrelation'],
]


Sensitivity comparisons registered: 49
                             Comparisons
AgreementOfConclusions                  
same significant conclusion           34
both non-significant                  12
t-test significant only                2
direction inconsistent                 1
